## Figure 7 / Figure S11 / Figure S12 Analysis Notebook

GitHub-facing notebook filtered against the updated Figure 7, Figure S11, and Figure S12 legends. Retained code-cell source text is unchanged; outputs/execution counts are stripped. Panels 7A, 7L, and 7M are not generated here because they are schematic/ATAC-LDSC analyses outside this R notebook.

### Panel Crosswalk

- Figure S11A-B and Figure 7B: motor-neuron nuclei UMAP and marker heatmap.
- Figure 7C: Wilcoxon rank-based DAMNDM enrichment summaries across ALS subtypes in alpha and gamma motor-neuron nuclei.
- Figure 7D-E and Figure S11C-E: DAMNDM score distributions, sample/donor means, and LMM statistics for nuclei.
- Figure 7F-G and Figure S12A-C: neuronal fragment UMAPs, marker overlays, and marker heatmaps.
- Figure S12D: laser-captured motor-neuron DAMNDM enrichment summary.
- Figure 7H: TDP-43 cryptic exon target heatmap across neuronal subtype contrasts.
- Figure 7I: Wilcoxon rank-based DAMNDM enrichment summaries across ALS subtypes in alphaFRAGs and gammaFRAGs.
- Figure 7J-K and Figure S12E-F: DAMNDM score distributions, sample/donor means, and LMM statistics for alphaFRAG/gammaFRAG/visceralFRAG populations.
- Figure S12G-J: alphaFRAG Nebula volcano plots for SOD1-ALS, C9-ALS, sALS, and pan-ALS versus control.


In [ ]:
# renv::init('/oak/stanford/groups/agitler/Shared/Shared_Jupyter_Notebook_Analysis/4.1.1-OG/')

## Setup And Paths

Load core R packages and define the Sherlock/Oak input and output paths used by the retained analyses.


In [ ]:
library(dplyr)
library(Seurat)
library(ggplot2)
library(ggrepel)
library(ComplexHeatmap)
# library(pheatmap)
library(gplots)
library(svglite)
library(hdf5r)
library(DropletUtils)
# library(scDblFinder)
library(scuttle)
library(scales)
library(ggrastr)
library(future)
plan("multicore", workers = availableCores())
options(future.globals.maxSize= +Inf)
# NotebookApp.max_buffer_size=214748364800

load.dir <- '/oak/stanford/groups/agitler/Shared/SOD1_Paper/Biogen_Shaolong_Data'
fig_dir <- '/oak/stanford/groups/agitler/Shared/SOD1_Paper/Biogen_Shaolong_Data/Debris_Figs'
csv_dir <- '/oak/stanford/groups/agitler/Shared/SOD1_Paper/Biogen_Shaolong_Data/CSV_Files'

In [ ]:
# Define your list of packages
pkgs <- c("nebula", "homologene", "lme4", "lmerTest", "emmeans")

# Get the matrix of all installed packages
all_installed <- installed.packages()

# Filter for the packages from your list that are currently installed
installed_pkgs <- intersect(pkgs, rownames(all_installed))

# Extract and print their names and version numbers
versions <- all_installed[installed_pkgs, "Version"]
print(versions)

In [ ]:
# Convert the named vector into a clean data frame
version_df <- data.frame(
  Package = names(versions),
  Version = as.character(versions),
  row.names = NULL
)

print(version_df)

## Publication Output Helpers

Define publication output directories and shared helpers for saving figure files.


In [ ]:
# --- 0. Directory & Function Setup --------------------------------------------
library(ggplot2)
library(dplyr)
library(ggpubr)
library(ggrepel)
library(homologene)
library(tibble)
library(Seurat)

# 1. Define Directory Structure
base_dir <- "/oak/stanford/groups/agitler/Shared/SOD1_Paper/Biogen_Shaolong_Data"
pub_dir  <- file.path(base_dir, "Publication_Figures_v1")

dirs <- list(
  concordance = file.path(pub_dir, "1_Concordance_Scatter"),
  dist        = file.path(pub_dir, "2_Score_Distributions"),
  stats       = file.path(pub_dir, "3_LMM_Stats"),
  drivers     = file.path(pub_dir, "4_Score_Drivers")
)

# Create directories recursively
lapply(dirs, dir.create, recursive = TRUE, showWarnings = FALSE)

# 2. Define Standardized Save Function
save_pub_plot <- function(plot_obj, filename_base, folder, w=6, h=6) {
  # Save Vector (PDF) - Editable
  ggsave(filename = file.path(folder, paste0(filename_base, ".pdf")), 
         plot = plot_obj, width = w, height = h, device = "pdf", useDingbats=FALSE)
  
  # Save Raster (PNG) - Quick View
  ggsave(filename = file.path(folder, paste0(filename_base, ".png")), 
         plot = plot_obj, width = w, height = h, dpi = 600)
  
  message(paste("Saved:", filename_base))
}

## H5AD Import And Integrity Audits

Load source H5AD content and audit metadata/count alignment before downstream Seurat analyses.


In [ ]:
library(hdf5r)
library(Seurat)
library(Matrix)

# --- 1. Setup -----------------------------------------------------------------
load.dir <- '/oak/stanford/groups/agitler/Shared/SOD1_Paper/Biogen_Shaolong_Data'
h5ad_file <- list.files(path = load.dir, pattern = "\\.h5ad$", full.names = TRUE)[1]

message(paste("Opening:", basename(h5ad_file)))
file.h5 <- H5File$new(h5ad_file, mode = "r")

# --- 2. Helper Functions ------------------------------------------------------
get_index_name <- function(group_obj) {
  if (group_obj$exists("_index")) return("_index")
  if (group_obj$attr_exists("_index")) return(group_obj$attr_open("_index")$read())
  if (group_obj$exists("index")) return("index")
  stop("Could not determine index column name.")
}

load_matrix <- function(file.h5, path, n_feat, n_cells) {
  if (!file.h5$exists(path)) return(NULL)
  obj <- file.h5[[path]]
  if (inherits(obj, "H5Group")) { # Sparse
    return(Matrix::sparseMatrix(i = obj[["indices"]][] + 1, p = obj[["indptr"]][], x = obj[["data"]][], dims = c(n_feat, n_cells)))
  } else { # Dense
    mat <- obj[,]; if (ncol(mat) != n_cells) mat <- t(mat)
    return(as(mat, "CsparseMatrix"))
  }
}

# --- 3. Matrix & Dimensions ---------------------------------------------------
message("Loading count matrix...")
use_raw <- file.h5$exists("raw/X")
matrix_path <- if (use_raw) "raw/X" else "X"
var_path <- if (use_raw) "raw/var" else "var"

obs_grp <- file.h5[["obs"]]
obs_idx <- get_index_name(obs_grp)
n_cells <- obs_grp[[obs_idx]]$dims
cell_names <- obs_grp[[obs_idx]][]

var_grp <- file.h5[[var_path]]
var_idx <- get_index_name(var_grp)
n_features <- var_grp[[var_idx]]$dims
feature_names <- var_grp[[var_idx]][]

message(paste0("Dimensions: ", n_features, " features x ", n_cells, " cells"))

counts <- load_matrix(file.h5, matrix_path, n_features, n_cells)
rownames(counts) <- feature_names
colnames(counts) <- cell_names

# --- 4. Robust Metadata Loader (The Fix) --------------------------------------
message("Extracting Metadata (with -1/NA fix)...")
obs_df <- data.frame(row.names = cell_names)
obs_keys <- names(obs_grp)

for (key in obs_keys) {
  # Skip indices and internal keys
  if (key %in% c("_index", obs_idx) || grepl("^__", key)) next
  
  tryCatch({
    obj <- obs_grp[[key]]
    
    # --- CASE A: Categorical (H5Group) ---
    if (inherits(obj, "H5Group") && obj$exists("codes") && obj$exists("categories")) {
      codes <- obj[["codes"]][]
      cats  <- obj[["categories"]][]
      
      final_vec <- rep(NA_character_, n_cells)
      valid_mask <- codes >= 0  # <--- CRITICAL FIX: Ignore -1 codes
      
      if (obj$exists("indices")) { # SPARSE
        indices <- obj[["indices"]][]
        valid_codes <- codes[valid_mask]
        valid_pos   <- indices[valid_mask] + 1
        final_vec[valid_pos] <- cats[valid_codes + 1]
      } else { # DENSE
        if (length(codes) == n_cells) {
           final_vec[valid_mask] <- cats[codes[valid_mask] + 1]
        }
      }
      obs_df[[key]] <- final_vec
    }
    
    # --- CASE B: Simple Dataset ---
    else if (inherits(obj, "H5D")) {
      raw_data <- obj[]
      if (is.matrix(raw_data) || is.array(raw_data)) raw_data <- as.vector(raw_data)
      
      if (length(raw_data) == n_cells) {
        obs_df[[key]] <- raw_data
      } else {
        # Safe padding
        final_vec <- rep(NA, n_cells)
        limit <- min(length(raw_data), n_cells)
        final_vec[1:limit] <- raw_data[1:limit]
        obs_df[[key]] <- final_vec
      }
    }
    
  }, error = function(e) {
    message(paste(" -> WARNING: Failed column", key, ":", e$message))
  })
}

# --- 5. Assemble Seurat Object ------------------------------------------------
message("Assembling Seurat Object...")
alphamn_obj <- CreateSeuratObject(counts = counts, meta.data = obs_df)

# Load UMAP if available
if (file.h5$exists("obsm/X_umap")) {
  message("Adding UMAP coordinates...")
  umap <- file.h5[["obsm/X_umap"]][,]
  if (nrow(umap) == 2 && ncol(umap) == n_cells) umap <- t(umap)
  rownames(umap) <- cell_names
  colnames(umap) <- c("UMAP_1", "UMAP_2")
  alphamn_obj[["umap"]] <- CreateDimReducObject(embeddings = umap, key = "UMAP_", assay = DefaultAssay(alphamn_obj))
}

file.h5$close_all()

# --- 6. Final Validation ------------------------------------------------------
message("\n✅ Load Complete!")
message("Metadata Summary (Status):")
print(table(alphamn_obj$status, useNA = "always"))

In [ ]:
library(hdf5r)
library(Seurat)

# --- Setup ---
load.dir <- '/oak/stanford/groups/agitler/Shared/SOD1_Paper/Biogen_Shaolong_Data'
h5ad_file <- list.files(path = load.dir, pattern = "\\.h5ad$", full.names = TRUE)[1]
file.h5 <- H5File$new(h5ad_file, mode = "r")

# --- TEST 1: Metadata Alignment (The "Off-by-One" Check) ---
# We will manually decode the first 10 cells of 'status' from the file
# and compare them to the first 10 cells in Seurat.

target_col <- "status" # Change to 'Diagnosis' or 'batch' if needed

if (file.h5$exists(paste0("obs/", target_col))) {
  message(paste("\n=== AUDIT 1: Checking Categorical Alignment ('", target_col, "') ===", sep=""))
  
  # A. Read Raw HDF5 Data
  grp <- file.h5[[paste0("obs/", target_col)]]
  
  if(grp$exists("codes") && grp$exists("categories")) {
    raw_codes <- grp[["codes"]][1:10]        # The integers (0, 1...)
    raw_cats  <- grp[["categories"]][]       # The labels
    
    # Python is 0-based, so code 0 -> Category 1. 
    # We manually simulate the mapping here:
    manual_decode <- raw_cats[raw_codes + 1]
    
    # B. Read Seurat Data
    seurat_values <- as.character(sobj@meta.data[[target_col]][1:10])
    
    # C. Compare
    print(data.frame(
      Cell_Index = 1:10,
      Raw_Code_H5 = raw_codes,
      Manual_Decode_H5 = manual_decode,
      Seurat_Value = seurat_values,
      MATCH = manual_decode == seurat_values
    ))
    
    if (all(manual_decode == seurat_values)) {
      message("-> SUCCESS: Categorical variables are aligned correctly.")
    } else {
      message("-> FAILURE: Mismatch detected! Metadata is scrambled.")
    }
  } else {
    message("Column exists but is not in 'codes/categories' format. Skipping deep check.")
  }
}

# --- TEST 2: Matrix Integrity (The "Counts vs Scaled" Check) ---
message("\n=== AUDIT 2: Checking Count Matrix Integrity ===")

# Get the raw counts from Seurat
counts_sample <- GetAssayData(sobj, layer = "counts")[1:10, 1:10]
min_val <- min(GetAssayData(sobj, layer = "counts"))
is_integer <- all(GetAssayData(sobj, layer = "counts")@x %% 1 == 0)

message(paste("Minimum value in Seurat 'counts':", min_val))
message(paste("Are all values integers?", is_integer))

if (min_val < 0) {
  message("-> CRITICAL FAILURE: 'counts' contains negative values!")
  message("   This means you loaded SCALED data (Scanpy .X) instead of RAW counts.")
  message("   Differential expression will be completely wrong.")
} else if (!is_integer) {
  message("-> WARNING: 'counts' contains decimals.")
  message("   This might be log-transformed data loaded as counts, or imputed data.")
  message("   Ensure you are using 'raw/X' from the h5ad file.")
} else {
  message("-> SUCCESS: 'counts' look like valid raw integer counts.")
}

# --- TEST 3: Feature Name Check ---
message("\n=== AUDIT 3: Gene Name Check ===")
# Check if the first gene in HDF5 matches the first gene in Seurat
# (Checks if we transposed the matrix correctly)

# Check H5AD var names
use_raw <- file.h5$exists("raw/var/_index")
var_path <- if(use_raw) "raw/var/_index" else "var/_index"
if(file.h5$exists(var_path)){
   h5_genes <- file.h5[[var_path]][1:5]
   seurat_genes <- rownames(sobj)[1:5]
   
   print(data.frame(H5_Gene = h5_genes, Seurat_Gene = seurat_genes, MATCH = h5_genes == seurat_genes))
}

file.h5$close_all()

In [ ]:
library(hdf5r)
library(Seurat)
library(Matrix)

# --- 1. Setup -----------------------------------------------------------------
load.dir <- '/oak/stanford/groups/agitler/Shared/SOD1_Paper/Biogen_Shaolong_Data'
h5ad_file <- list.files(path = load.dir, pattern = "\\.h5ad$", full.names = TRUE)[1]

message(paste("Opening:", basename(h5ad_file)))
file.h5 <- H5File$new(h5ad_file, mode = "r")

# --- 2. Helper Functions (PRESERVED) ------------------------------------------
get_index_name <- function(group_obj) {
  if (group_obj$exists("_index")) return("_index")
  if (group_obj$attr_exists("_index")) return(group_obj$attr_open("_index")$read())
  if (group_obj$exists("index")) return("index")
  stop("Could not determine index column name.")
}

# --- 3. Robust Metadata Loader (FIXED) ----------------------------------------
message("Extracting Metadata...")

# A. Get Cell Names (The Anchor)
obs_grp <- file.h5[["obs"]]
obs_idx <- get_index_name(obs_grp)
n_cells <- obs_grp[[obs_idx]]$dims
cell_names <- obs_grp[[obs_idx]][]

message(paste("Total Cells:", n_cells))

# B. Initialize Dataframe
obs_df <- data.frame(row.names = cell_names)
obs_keys <- names(obs_grp)

for (key in obs_keys) {
  # Skip indices and internal keys
  if (key %in% c("_index", obs_idx) || grepl("^__", key)) next
  
  tryCatch({
    obj <- obs_grp[[key]]
    
    # --- CASE A: Categorical (H5Group with codes/categories) ---
    if (inherits(obj, "H5Group") && obj$exists("codes") && obj$exists("categories")) {
      
      # 1. Load Raw Codes
      codes <- obj[["codes"]][]
      cats  <- obj[["categories"]][]
      
      # 2. FIX: Handle -1 (NA) codes
      # Python uses -1 for NA. We must convert these to NA before indexing.
      # Create a vector of NAs with the correct length
      final_vec <- rep(NA_character_, n_cells)
      
      # Identify valid indices (>= 0)
      valid_mask <- codes >= 0
      
      # 3. Handle Indexing
      if (obj$exists("indices")) {
        # SPARSE Case
        indices <- obj[["indices"]][] # Locations of the data
        # Only map the valid codes
        # Note: 'codes' here matches the length of 'indices', not n_cells
        valid_codes <- codes[valid_mask]
        valid_pos   <- indices[valid_mask] + 1 # 0-based -> 1-based
        
        final_vec[valid_pos] <- cats[valid_codes + 1]
        
      } else {
        # DENSE Case (This was failing before)
        # Verify length matches n_cells
        if (length(codes) == n_cells) {
           # Map valid codes to categories
           # We only update positions where code != -1
           final_vec[valid_mask] <- cats[codes[valid_mask] + 1]
        } else {
           warning(paste("Length mismatch in", key, "- Skipping."))
        }
      }
      
      # Assign to dataframe
      obs_df[[key]] <- final_vec
    }
    
    # --- CASE B: Simple Dataset ---
    else if (inherits(obj, "H5D")) {
      raw_data <- obj[]
      
      # Handle dimensions (sometimes 1D arrays are stored as N x 1 matrices)
      if (is.matrix(raw_data) || is.array(raw_data)) raw_data <- as.vector(raw_data)
      
      if (length(raw_data) == n_cells) {
        obs_df[[key]] <- raw_data
      } else {
        # Only pad if strictly necessary (shouldn't happen for dense columns)
        final_vec <- rep(NA, n_cells)
        limit <- min(length(raw_data), n_cells)
        final_vec[1:limit] <- raw_data[1:limit]
        obs_df[[key]] <- final_vec
      }
    }
    
  }, error = function(e) {
    message(paste(" -> WARNING: Failed column", key, ":", e$message))
  })
}

file.h5$close_all()

# --- 4. Apply Fix to Existing Object ------------------------------------------
message("\nApplying repaired metadata to 'sobj'...")

# We ensure alignment by matching row names
# This is safer than just replacing @meta.data
sobj <- AddMetaData(sobj, metadata = obs_df)

message("Success! Metadata repaired.")
message("Check 'status' column NAs (should now be valid missing data, not alignment errors):")
print(table(sobj$status, useNA = "always"))

## Integrated Object Construction

Build or load the integrated single-nucleus/droplet object used as the base for nuclei and fragment analyses.


In [ ]:
# --- 1. Create Composite Metadata Column --------------------------------------
# Ensure columns are characters to avoid Factor integer pasting
s_id <- as.character(sobj$SampleID)
batch <- as.character(sobj$batch)
tissue <- as.character(sobj$tissue)

# Replace any NAs with "Unknown" to ensure clean grouping
s_id[is.na(s_id)] <- "UnknownID"
batch[is.na(batch)] <- "UnknownBatch"
tissue[is.na(tissue)] <- "UnknownTissue"

# Create the new column
sobj$integration_group <- paste(s_id, batch, tissue, sep = "_")

# Verify the groups
message(paste("Created", length(unique(sobj$integration_group)), "unique integration groups."))
print(table(sobj$integration_group))
library(Seurat)

library(Seurat)
library(harmony) # Ensure this is installed: install.packages("harmony")
library(future)

# --- 1. Setup & Speed Options -------------------------------------------------
plan("multicore", workers = 12)
options(future.globals.maxSize = 8000 * 1024^2)

# --- 2. Filter Tiny Groups ----------------------------------------------------
# Harmony is robust, but filtering < 15 cells is still good practice for PCA stability
group_counts <- table(sobj$integration_group)
tiny_groups <- names(group_counts[group_counts < 20])

if (length(tiny_groups) > 0) {
  message("Removing ", length(tiny_groups), " groups with < 20 cells...")
  sobj <- subset(sobj, subset = integration_group %in% tiny_groups, invert = TRUE)
}
library(Seurat)
library(harmony)
library(future)



# --- 2. Robust Feature Selection (The Fix) ------------------------------------
# A. JOIN LAYERS FIRST
# We combine all data to calculate Variable Features robustly. 
# This prevents LOESS errors on small batches.
sobj[["RNA"]] <- JoinLayers(sobj[["RNA"]])

# B. Normalize & Find Features Globally
message("Normalizing and Finding Features (Global)...")
sobj <- NormalizeData(sobj, verbose = FALSE)
sobj <- FindVariableFeatures(sobj, selection.method = "vst", nfeatures = 2000, verbose = FALSE)

# --- 3. Prepare for Harmony ---------------------------------------------------
# Now that we have our 2000 features, we SPLIT the object back up.
# This ensures Harmony sees the batches as distinct groups for integration.
message("Splitting layers for Integration...")
sobj[["RNA"]] <- split(sobj[["RNA"]], f = sobj$integration_group)

# --- 4. Scale Data (Per Batch) ------------------------------------------------
# We scale data on the split layers. This performs "Batch Scaling" (Z-scoring within each batch),
# which is the first step of batch correction.
message("Scaling Data (Parallelized)...")
sobj <- ScaleData(sobj, verbose = FALSE)

# --- 5. Run PCA ---------------------------------------------------------------
message("Running PCA...")
sobj <- RunPCA(sobj, 
               npcs = 20, # 15-20 is plenty for this data
               features = VariableFeatures(sobj), 
               approx = TRUE, 
               verbose = FALSE)

# --- 6. Run Harmony -----------------------------------------------------------
message("Running Harmony Integration...")
sobj <- IntegrateLayers(
  object = sobj, 
  method = HarmonyIntegration,
  orig.reduction = "pca", 
  new.reduction = "harmony",
  dims = 1:20,
  verbose = FALSE
)

# --- 7. Clustering & Saving ---------------------------------------------------
message("Clustering...")
sobj <- FindNeighbors(sobj, reduction = "harmony", dims = 1:20, verbose = FALSE)
sobj <- FindClusters(sobj, resolution = 0.5, verbose = FALSE)
sobj <- RunUMAP(sobj, dims = 1:20, reduction = "harmony", verbose = FALSE)

# Re-join layers for final object structure
sobj <- JoinLayers(sobj)

plan("sequential")

# Save
saveRDS(sobj, file = file.path(load.dir, "SOD1_Integrated_Harmony.rds"))
message("Success! Object saved.")

# Plot


In [ ]:
sobj <- readRDS(file.path(load.dir, "SOD1_Integrated_Harmony.rds"))


## Axoplasmic Fragment Object Construction

Subset and reprocess axoplasmic neuronal fragments before final subtype annotation.


In [ ]:
library(Seurat)
library(harmony)
library(dplyr)
library(patchwork)

# --- 1. Parameters ------------------------------------------------------------
# Extended list of all axoplasmic clusters
target_clusters <- c("1", "8", "25", "14", "18", "23", "7", "21", "20", "17", "19", "2", "11", "12", "16", "4")
min_cells_threshold <- 5   # Minimum cells per batch to keep

# --- 2. Subset to Target Clusters ---------------------------------------------
message(paste("Subsetting to axoplasmic clusters:", paste(target_clusters, collapse=","), "..."))
axo_obj <- subset(sobj, idents = target_clusters)

# Important: Join layers immediately to clean up empty/fragmented layers
axo_obj[["RNA"]] <- JoinLayers(axo_obj[["RNA"]])

# --- 3. Filter Small Batches --------------------------------------------------
# Check batch counts in this specific sub-population
batch_counts <- table(axo_obj$integration_group)

# Identify valid batches
valid_batches <- names(batch_counts[batch_counts >= min_cells_threshold])
removed_batches <- names(batch_counts[batch_counts < min_cells_threshold])

message(paste("Keeping", length(valid_batches), "batches."))
if (length(removed_batches) > 0) {
  message(paste("Removing", length(removed_batches), "batches with <", min_cells_threshold, "cells."))
  axo_obj <- subset(axo_obj, subset = integration_group %in% valid_batches)
}

# --- 4. Re-Processing (Normalize -> PCA) --------------------------------------
message("Re-normalizing and finding variable features...")
library(Seurat)
library(Matrix)

DefaultAssay(axo_obj) <- "RNA"

mito_genes   <- c(grep("^MT-", rownames(axo_obj), value = TRUE),grep("^MTRN", rownames(axo_obj), value = TRUE))   # human; mouse often "^mt-"
nonmito_genes <- setdiff(rownames(axo_obj), mito_genes)

counts_nonmito <- GetAssayData(axo_obj, assay = "RNA", slot = "counts")[nonmito_genes, , drop = FALSE]

axo_obj[["RNA_noMT"]] <- CreateAssayObject(counts = counts_nonmito)
DefaultAssay(axo_obj) <- "RNA_noMT"

axo_obj <- NormalizeData(axo_obj, normalization.method = "LogNormalize", scale.factor = 1e4)
axo_obj <- FindVariableFeatures(axo_obj)
axo_obj <- ScaleData(axo_obj)

message("Running PCA...")
# Increased NPCs to 30 to capture diversity of multiple neuron types
axo_obj <- RunPCA(axo_obj, npcs = 30, verbose = FALSE) 

# --- 5. Re-Harmonize ----------------------------------------------------------
# To run IntegrateLayers, we must split the object by batch again
message("Splitting layers for Harmony...")
axo_obj[["RNA_noMT"]] <- split(axo_obj[["RNA_noMT"]], f = axo_obj$integration_group)

message("Running Harmony Integration...")
axo_obj <- IntegrateLayers(
  object = axo_obj, 
  method = HarmonyIntegration,
  orig.reduction = "pca", 
  new.reduction = "harmony",
  dims = 1:30,
  verbose = FALSE
)

# --- 6. Re-Cluster and UMAP ---------------------------------------------------
message("Re-clustering...")
axo_obj <- FindNeighbors(axo_obj, reduction = "harmony", dims = 1:30, verbose = FALSE)
axo_obj <- FindClusters(axo_obj, resolution = 1, verbose = FALSE) 
axo_obj <- RunUMAP(axo_obj, dims = 1:30, reduction = "harmony", verbose = FALSE)

# Clean up layers for easy plotting
axo_obj <- JoinLayers(axo_obj)

# --- 7. Visualization ---------------------------------------------------------
message("Done!")

# Plot 1: The new sub-clusters
p1 <- DimPlot(axo_obj, reduction = "umap", label = TRUE) + 
      ggtitle("Sub-clustering of All Axoplasmic Debris")

# Plot 2: Check integration (should look mixed)
p2 <- DimPlot(axo_obj, reduction = "umap", group.by = "integration_group") + 
      NoLegend() + ggtitle("Batch Integration Check")

# Display side-by-side
print(p1 + p2)

# Save the object for safety
saveRDS(axo_obj, file = file.path(load.dir, "Axoplasmic_Debris_Subclustered.rds"))

In [ ]:
axo_obj <- readRDS(file.path(load.dir, "Axoplasmic_Debris_Subclustered.rds"))

## Motor Neuron Nuclei Reference

Load the motor-neuron nuclei reference used for Figure 7B and Figure S11A-B.


In [ ]:
nuclei_mn <- readRDS("/oak/stanford/groups/agitler/Shared/SOD1_Paper/Biogen_Shaolong_Data/MN_sub_nFeature_3361_final.rds")

## Figure 7B / Figure S11A-B: Nuclei Subtypes

Generate nuclei subtype UMAP and marker heatmap outputs used to annotate alpha, gamma, gamma-star, ATF3-positive alpha, and visceral motor-neuron nuclei.


In [ ]:
# --- 0. Directory & Library Setup ---------------------------------------------
library(Seurat)
library(ggplot2)
library(dplyr)
library(ggrastr) 
library(colorspace) 
library(ggrepel) # Essential for the halo/repulsion

if(!exists("pub_dir")) pub_dir <- "Plots" 
if(!exists("dirs")) dirs <- list()
dirs$mn_subtypes <- file.path(pub_dir, "MN_Subtypes_UMAP")
dir.create(dirs$mn_subtypes, recursive = TRUE, showWarnings = FALSE)

# --- 1. Define Metadata & Colors ----------------------------------------------
# 1. Create Metadata (Updated labels)
nuclei_mn$MN_Type <- as.character(nuclei_mn$seurat_clusters)
nuclei_mn$MN_Type[nuclei_mn$seurat_clusters %in% c(11)] <- "ATF3+ Alpha MNs"
nuclei_mn$MN_Type[nuclei_mn$seurat_clusters %in% c(1, 4)] <- "Alpha MNs"
nuclei_mn$MN_Type[nuclei_mn$seurat_clusters %in% c(0, 8)]       <- "Gamma MNs"
nuclei_mn$MN_Type[nuclei_mn$seurat_clusters %in% c(10)]         <- "Gamma* MNs"
nuclei_mn$MN_Type[nuclei_mn$seurat_clusters %in% c(2, 3, 5, 6, 7, 9)] <- "Visceral MNs"

nuclei_mn$MN_Type <- factor(nuclei_mn$MN_Type, 
                            levels = c("Alpha MNs", "Gamma MNs", "Gamma* MNs", "ATF3+ Alpha MNs", "Visceral MNs"))
Idents(nuclei_mn) <- "MN_Type"

# 2. Define Base Palette (Updated palette to match heatmap)
my_mn_colors <- c(
  "Alpha MNs"           = "#E41A1C", 
  "ATF3+ Alpha MNs"  = "#FB9A99", 
  "Gamma MNs"           = "#4DAF4A", 
  "Gamma* MNs" = "#006D2C", 
  "Visceral MNs"        = "#377EB8"  
)

# 3. Define Darker Text Palette (30% darker for contrast in v1)
text_colors <- darken(my_mn_colors, amount = 0.3)
names(text_colors) <- names(my_mn_colors)

# --- 2. Calculate Centroids (Two Sets for Dual Plotting) ----------------------
# Extract UMAP coordinates
umap_coords <- Embeddings(nuclei_mn, "umap") %>% as.data.frame()
colnames(umap_coords)[1:2] <- c("UMAP_1", "UMAP_2")
umap_coords$MN_Type <- nuclei_mn$MN_Type
umap_coords$seurat_clusters <- nuclei_mn$seurat_clusters

# Centroids for broad MN Types (Used in v1)
label_data_type <- umap_coords %>%
  group_by(MN_Type) %>%
  summarise(UMAP_1 = median(UMAP_1), UMAP_2 = median(UMAP_2), .groups = 'drop')

# Centroids for specific Seurat clusters (Used in v2 and v3)
label_data_cluster <- umap_coords %>%
  group_by(seurat_clusters) %>%
  summarise(UMAP_1 = median(UMAP_1), UMAP_2 = median(UMAP_2), .groups = 'drop')

# --- 3. Base Plot Setup (MN_Type Grouping - Used for v1 & v2) -----------------
p_umap_clean <- DimPlot(nuclei_mn, 
                        reduction = "umap", 
                        group.by = "MN_Type",
                        cols = my_mn_colors,
                        pt.size = 1.5,
                        label = FALSE, 
                        raster = FALSE) + 
  theme_void() + 
  theme(legend.position = "none", plot.title = element_blank())

# Rasterize Points
if (length(p_umap_clean$layers) > 0) {
  p_umap_clean$layers[[1]] <- rasterise(p_umap_clean$layers[[1]], dpi = 300)
}

# --- 4. Visualization 1: Original Style (MN Types Only) -----------------------
p_umap_v1 <- p_umap_clean + 
  geom_text_repel(data = label_data_type, 
                  aes(x = UMAP_1, y = UMAP_2, label = MN_Type, color = MN_Type),
                  fontface = "bold", 
                  size = 6,
                  bg.color = "white", bg.r = 0.15,        
                  segment.color = NA, 
                  force = 5,          
                  seed = 42) +
  scale_color_manual(values = text_colors)

# --- 5. Visualization 2: Numbers on Plot, Types in Legend ---------------------
p_umap_v2 <- p_umap_clean + 
  geom_text_repel(data = label_data_cluster, 
            aes(x = UMAP_1, y = UMAP_2, label = seurat_clusters),
            fontface = "bold", 
            size = 6,
            color = "black",          
            bg.color = "white",       
            bg.r = 0.15,              
            segment.color = NA, 
            force = 0,                
            seed = 42) +
  theme(legend.position = "right",
        legend.title = element_blank(),
        legend.text = element_text(size = 16, face = "bold"),
        legend.margin = margin(t = 0, r = 0, b = 0, l = 30)) + 
  guides(color = guide_legend(override.aes = list(size = 5)))

# --- 6. Visualization 3: Strictly seurat_clusters Aesthetic -------------------
# Create a new base plot grouped strictly by clusters using default Seurat colors
p_umap_clusters_base <- DimPlot(nuclei_mn, 
                        reduction = "umap", 
                        group.by = "seurat_clusters",
                        pt.size = 1.5,
                        label = FALSE, 
                        raster = FALSE) + 
  theme_void() + 
  theme(legend.position = "none", plot.title = element_blank())

# Rasterize the cluster base plot
if (length(p_umap_clusters_base$layers) > 0) {
  p_umap_clusters_base$layers[[1]] <- rasterise(p_umap_clusters_base$layers[[1]], dpi = 300)
}

# Add the halo labels to the clusters
p_umap_v3 <- p_umap_clusters_base + 
  geom_text_repel(data = label_data_cluster, 
            aes(x = UMAP_1, y = UMAP_2, label = seurat_clusters),
            fontface = "bold", 
            size = 6,                 # Slightly larger text since there's no legend
            color = "black",          # Black text ensures contrast against the rainbow points
            bg.color = "white",       
            bg.r = 0.15,              
            segment.color = NA, 
            force = 0,                
            seed = 42)

# --- 7. Save Outputs ----------------------------------------------------------
out_name_v1 <- "MN_Subtypes_UMAP_TypesOnly"
out_name_v2 <- "MN_Subtypes_UMAP_ClustersWithLegend"
out_name_v3 <- "MN_Subtypes_UMAP_SeuratClusters"

if(exists("save_pub_plot")) {
  save_pub_plot(p_umap_v1, out_name_v1, dirs$mn_subtypes, w=6, h=6)
  save_pub_plot(p_umap_v2, out_name_v2, dirs$mn_subtypes, w=7, h=6) 
  save_pub_plot(p_umap_v3, out_name_v3, dirs$mn_subtypes, w=6, h=6) 
} else {
  # Save Plot 1
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_v1, ".pdf")), p_umap_v1, width=8, height=6)
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_v1, ".png")), p_umap_v1, width=8, height=6, dpi=600)
  
  # Save Plot 2
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_v2, ".pdf")), p_umap_v2, width=9, height=6)
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_v2, ".png")), p_umap_v2, width=9, height=6, dpi=600)

  # Save Plot 3
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_v3, ".pdf")), p_umap_v3, width=8, height=6)
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_v3, ".png")), p_umap_v3, width=8, height=6, dpi=600)
}

message("Plots saved to: ", dirs$mn_subtypes)

In [ ]:
nuclei_mn <- NormalizeData(nuclei_mn, 
                           normalization.method = "LogNormalize", 
                           scale.factor = 10000)

In [ ]:
# --- 0. Directory & Library Setup ---------------------------------------------
library(Seurat)
library(dplyr)
library(pheatmap)
library(matrixStats) # Added for rowMins() and rowMaxs()

if(!exists("pub_dir")) pub_dir <- "Plots" 
if(!exists("dirs")) dirs <- list()
dirs$mn_subtypes <- file.path(pub_dir, "MN_Subtypes_Heatmap")
dir.create(dirs$mn_subtypes, recursive = TRUE, showWarnings = FALSE)

# --- 1. Heatmap: Calculation (Depth-Normalized -> Min/Max Scaled) -------------
# Define Cholinergic Subtype Markers
alpha_markers    <- c("VIPR2", "PEX5L")
gamma_markers    <- c("AMTN", "IL33")
visceral_markers <- c("NOS1", "FBN2", "ZEB2")
other_markers    <- c("RORA","ESRRG", "BCL6", "KIRREL3", "ATF3") 

heatmap_features <- c(alpha_markers, gamma_markers, visceral_markers, other_markers)

# Calculate Average Expression across seurat_clusters (Depth-Normalized)
# Note: avg_exp has genes as rows and clusters as columns
avg_exp <- AverageExpression(nuclei_mn, features = heatmap_features, group.by = "seurat_clusters", slot = 'data')$RNA
avg_exp <- as.matrix(avg_exp)

# Min-Max Scale per gene (row) matching your reference code
mat_scaled <- (avg_exp - rowMins(avg_exp)) / (rowMaxs(avg_exp) - rowMins(avg_exp))

# Safety net: If a gene has identical expression across all clusters (max == min), 
# the formula divides by 0 and creates NaNs. This forces them to 0.
mat_scaled[is.na(mat_scaled)] <- 0

# Transpose so clusters are rows and genes are columns (matching your original orientation)
mat_plot <- t(mat_scaled)

# --- 2. Define Cluster Annotations & Colors (Dynamically matched) -------------
# Pull exact row names from the matrix to guarantee pheatmap accepts them
cluster_levels <- rownames(mat_plot)
cluster_mapping <- data.frame(Cluster = cluster_levels, row.names = cluster_levels)

# Strip out any hidden characters/letters Seurat might have added to cluster names for matching
clean_clusters <- gsub("[^0-9]", "", cluster_levels)

cluster_mapping$MN_Type <- "Unknown"
cluster_mapping$MN_Type[clean_clusters %in% c("11")] <- "ATF3+ Alpha MNs"
cluster_mapping$MN_Type[clean_clusters %in% c("1", "4")] <- "Alpha MNs"
cluster_mapping$MN_Type[clean_clusters %in% c("0", "8")]        <- "Gamma MNs"
cluster_mapping$MN_Type[clean_clusters %in% c("10")]            <- "Gamma* MNs"
cluster_mapping$MN_Type[clean_clusters %in% c("2", "3", "5", "6", "7", "9")] <- "Visceral MNs"

# Format as an annotation dataframe for pheatmap
annotation_row <- data.frame(MN_Type = factor(cluster_mapping$MN_Type, 
                             levels = c("Alpha MNs", "Gamma MNs", "Gamma* MNs", "ATF3+ Alpha MNs", "Visceral MNs")))
rownames(annotation_row) <- rownames(cluster_mapping)

# Define matching colors for the annotation sidebar
my_mn_colors <- c(
  "Alpha MNs"           = "#E41A1C", # Original Alpha Red
  "ATF3+ Alpha MNs"  = "#FB9A99", # Lighter Red/Salmon for ATF3+ Alphas
  "Gamma MNs"           = "#4DAF4A", # Original Gamma Green
  "Gamma* MNs" = "#006D2C", # Original Gamma* Dark Green
  "Visceral MNs"        = "#377EB8"  # Original Visceral Blue
)
ann_colors <- list(MN_Type = my_mn_colors)

# --- 3. Visualization: Annotated Heatmap --------------------------------------
pdf_path_hm <- file.path(dirs$mn_subtypes, "Heatmap_MN_Subtypes.pdf")
png_path_hm <- file.path(dirs$mn_subtypes, "Heatmap_MN_Subtypes.png")

heatmap_args <- list(
  mat = mat_plot,          
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "none",          # Kept "none" because we already manually scaled it
  border_color = "white",  # Note: Removed the custom color arg to use pheatmap's default scheme
  
  # Add the MN_Type metadata annotations and colors
  annotation_row = annotation_row,
  annotation_colors = ann_colors,
  
  # Aesthetics
  fontsize = 13,           
  fontsize_row = 13,        
  fontsize_col = 13,       
  treeheight_row = 20,     
  treeheight_col = 20,     
  angle_col = 90,
  main = NA                
)

# Save Outputs
do.call(pheatmap::pheatmap, c(heatmap_args, list(filename = pdf_path_hm, width = 8.6, height = 4.34)))
do.call(pheatmap::pheatmap, c(heatmap_args, list(filename = png_path_hm, width = 8.6, height = 4.34)))
message("Saved Annotated Heatmap to: ", dirs$mn_subtypes)

## Figure 7C/I And Figure S12D: DAMNDM Rank-Based Enrichment

Prepare DAMNDM gene sets, generate per-comparison Wilcoxon/Hodges-Lehmann statistics, and render FDR-corrected median-shift summary plots for nuclei, fragments, and laser-captured motor neurons.


In [ ]:
library(dplyr)
library(ggplot2)
library(tibble)
library(homologene)
library(fgsea)

# --- 0. Directory Setup ---
if(!exists("pub_dir")) pub_dir <- "Plots"
dirs$gsea_pub <- file.path(pub_dir, "10_GSEA_Publication_Style")
dir.create(dirs$gsea_pub, recursive = TRUE, showWarnings = FALSE)

message(paste0("Output Directory: ", normalizePath(dirs$gsea_pub)))

# --- PARAMETERS ---
TOP_N <- 1000          # Max genes to take
MOUSE_P_THRESH <- 0.01 # Relaxed threshold

# --- 1. Prepare Gene Sets (Safeguarded) ---
if(file.exists("damn_signature.csv")) {
  mouse_raw <- read.csv("damn_signature.csv", stringsAsFactors = FALSE)
  if(!"gene" %in% colnames(mouse_raw)) colnames(mouse_raw)[1] <- "gene"

  # Map to Human
  hg <- homologene(unique(mouse_raw$gene), inTax = 10090, outTax = 9606)
  mm2hs <- hg %>% as_tibble() %>% dplyr::rename(mouse_gene = `10090`, human_gene = `9606`)

  mouse_sig <- mouse_raw %>%
    inner_join(mm2hs, by = c("gene" = "mouse_gene")) %>%
    transmute(human_gene, mouse_lfc = log2FoldChange, mouse_padj = padj) %>%
    filter(!is.na(human_gene), !is.na(mouse_padj)) %>% 
    distinct(human_gene, .keep_all = TRUE) 

  # --- SAFEGUARDED SELECTION ---
  genes_up <- mouse_sig %>% 
    filter(mouse_lfc > 0, mouse_padj < MOUSE_P_THRESH) %>% 
    arrange(desc(mouse_lfc)) %>% 
    head(TOP_N) %>% pull(human_gene)

  genes_down <- mouse_sig %>% 
    filter(mouse_lfc < 0, mouse_padj < MOUSE_P_THRESH) %>% 
    arrange(mouse_lfc) %>% 
    head(TOP_N) %>% pull(human_gene)
    
  damn_pathways <- list()
  if(length(genes_up) >= 5) damn_pathways[["DM Up"]] <- genes_up
  if(length(genes_down) >= 5) damn_pathways[["DM Down"]] <- genes_down
  
  message(paste0("Gene Sets Prepared:"))
  message(paste0("  > DM Up:   ", length(genes_up), " genes"))
  message(paste0("  > DM Down: ", length(genes_down), " genes"))
  
  if(length(damn_pathways) == 0) stop("Error: No significant mouse genes found.")

} else { stop("Error: damn_signature.csv not found") }



In [ ]:
# --- 2. Custom Plotting Function (Updated with Wilcoxon CIs) ---
run_gsea_pub_style <- function(res_df, genotype_label, cell_type, output_dir) {
  
  message(paste0("Running Enrichment Analysis: ", cell_type, " - ", genotype_label))
  
# A. Prepare Ranks (Universal metric handling)
  res_clean <- as.data.frame(res_df)
  
  # Ensure we have a gene identifier column
  # FIX: Use quotes and standard base R to safely rename without dplyr scoping errors
  if("gene_name" %in% colnames(res_clean)) {
    colnames(res_clean)[colnames(res_clean) == "gene_name"] <- "gene"
  } else if(!"gene" %in% colnames(res_clean)) {
    # If no 'gene' or 'gene_name' column exists, assume the rownames are the genes
    res_clean <- rownames_to_column(res_clean, var = "gene")
  }
  
  # Check if a pre-calculated 'stat' column already exists (e.g., DESeq2 Wald stat)
  if("stat" %in% colnames(res_clean)) {
    rank_df <- res_clean %>%
      filter(!is.na(stat), !is.na(gene), gene != "") %>%
      group_by(gene) %>% summarize(stat = mean(stat)) %>% ungroup() %>%
      arrange(desc(stat))
      
  } else {
    # If no 'stat' column, calculate it manually from pval and LFC 
    rank_df <- res_clean %>%
      filter(!is.na(pvalue), !is.na(log2FoldChange), !is.na(gene), gene != "") %>%
      mutate(stat = -log10(pvalue) * sign(log2FoldChange)) %>%
      mutate(stat = ifelse(is.infinite(stat), sign(stat) * max(abs(stat[is.finite(stat)])), stat)) %>%
      group_by(gene) %>% summarize(stat = mean(stat)) %>% ungroup() %>%
      arrange(desc(stat))
  }
  
  ranks <- rank_df$stat
  names(ranks) <- rank_df$gene
    
# B. === CORRECTED: RUN WILCOXON RANK-SUM TESTS (WITH TWO-SIDED CIs) ===
  wilcox_res <- list()
  for(pathway_name in names(damn_pathways)) {
    pathway_genes <- damn_pathways[[pathway_name]]
    
    # Split the dataset into "In Pathway" and "Background"
    in_pathway_stats  <- ranks[names(ranks) %in% pathway_genes]
    out_pathway_stats <- ranks[!names(ranks) %in% pathway_genes]
    
    if(length(in_pathway_stats) >= 5) {
      
      # 1. We MUST run a two-sided test to get a bounded Confidence Interval
      wt_two_sided <- wilcox.test(x = in_pathway_stats, y = out_pathway_stats, 
                                  alternative = "two.sided", exact = FALSE, conf.int = TRUE)
      
      # Extract the formal Hodges-Lehmann median shift estimate and bounds
      median_shift <- wt_two_sided$estimate
      ci_low  <- wt_two_sided$conf.int[1]
      ci_high <- wt_two_sided$conf.int[2]
      
      # 2. Convert to a ONE-SIDED p-value based on our biological expectation
      # A two-sided p-value tests "Is it different?". A one-sided tests "Did it go the right way?"
      two_sided_p <- wt_two_sided$p.value
      
      if (pathway_name == "DM Up") {
        # If it shifted right (positive), the one-sided p-val is half the two-sided.
        # If it shifted wrong (negative), the one-sided p-val is effectively 1.0
        final_p_val <- ifelse(median_shift > 0, two_sided_p / 2, 1.0)
        
      } else if (pathway_name == "DM Down") {
        # If it shifted left (negative), the one-sided p-val is half the two-sided.
        final_p_val <- ifelse(median_shift < 0, two_sided_p / 2, 1.0)
      }
      
      # Store results
      wilcox_res[[pathway_name]] <- list(
        p_val = final_p_val,
        median_shift = median_shift,
        ci_low = ci_low,
        ci_high = ci_high
      )
    }
  }
  # ====================================================================

  # C. Run fgsea (Used to generate the curves and NES)
  tryCatch({
    
    fgseaRes <- fgsea(pathways = damn_pathways, 
                      stats    = ranks,
                      minSize  = 5,      
                      maxSize  = 3000, 
                      nPermSimple = 10000)
    
    # === NEW: EXPORT COMBINED STATS TO CSV ===
    if (nrow(fgseaRes) > 0) {
      out_table <- fgseaRes %>% as_tibble()
      
      # Append the Wilcoxon stats and CIs to the GSEA output table
      out_table$Wilcoxon_Pval <- sapply(out_table$pathway, function(p) ifelse(!is.null(wilcox_res[[p]]), wilcox_res[[p]]$p_val, NA))
      out_table$Median_Shift <- sapply(out_table$pathway, function(p) ifelse(!is.null(wilcox_res[[p]]), wilcox_res[[p]]$median_shift, NA))
      out_table$ci_low <- sapply(out_table$pathway, function(p) ifelse(!is.null(wilcox_res[[p]]), wilcox_res[[p]]$ci_low, NA))
      out_table$ci_high <- sapply(out_table$pathway, function(p) ifelse(!is.null(wilcox_res[[p]]), wilcox_res[[p]]$ci_high, NA))
      
      out_table <- out_table %>%
        mutate(leadingEdge = sapply(leadingEdge, paste, collapse = "; ")) %>% 
        arrange(Wilcoxon_Pval) # Sort by Wilcoxon significance now
      
      csv_name <- paste0("Enrichment_Stats_", cell_type, "_", genotype_label, ".csv")
      write.csv(out_table, file.path(output_dir, csv_name), row.names = FALSE)
      message(paste0("    Saved CSV: ", csv_name))
    }
    # =========================================

    # D. Plot Each Pathway
    for(pathway_name in names(damn_pathways)) {
      
      if(pathway_name %in% fgseaRes$pathway) {
        
        stats_row <- fgseaRes[pathway == pathway_name]
        nes       <- stats_row$NES
        wilcox_p  <- wilcox_res[[pathway_name]]$p_val
        
        # Color logic driven by Wilcoxon
        line_color <- "grey80" 
        if (!is.null(wilcox_p) && wilcox_p < 0.01) {
          if (!is.na(nes) && nes > 0) line_color <- "#D53E4F" 
          if (!is.na(nes) && nes < 0) line_color <- "#3288BD" 
        }
        
        # Extract Curve Data
        base_plot <- plotEnrichment(damn_pathways[[pathway_name]], ranks)
        curve_data <- base_plot$data 
        if("rank" %in% colnames(curve_data)) curve_data$x <- curve_data$rank
        if("ES" %in% colnames(curve_data))   curve_data$y <- curve_data$ES
        
        # Subtitle combines both metrics
        subtitle_txt <- "Stats unavailable"
        if(!is.na(nes) & !is.null(wilcox_p)) {
          subtitle_txt <- paste0("NES = ", round(nes, 2), " | Wilcox P = ", formatC(wilcox_p, format = "e", digits = 1))
        }
        
        p <- ggplot(curve_data, aes(x = x, y = y)) +
          geom_hline(yintercept = 0, color = "black", linewidth = 0.5) +
          geom_line(color = line_color, linewidth = 1.5) +
          labs(
            title = paste0(genotype_label, " - ", pathway_name),
            subtitle = subtitle_txt,
            x = "Ranked genes (Up - Down)",
            y = "Enrichment Score"
          ) +
          theme_classic(base_size = 12) + 
          theme(
            plot.title    = element_text(face = "bold", hjust = 0.5, size = 12),
            plot.subtitle = element_text(hjust = 0.5, size = 10, margin = margin(b = 10)),
            axis.text     = element_text(color = "black", size = 10),
            axis.title    = element_text(face = "plain", size = 12),
            axis.line     = element_line(linewidth = 0.8) 
          )
        
        safe_pathway <- gsub("_", "", pathway_name)
        fname <- paste0("Enrichment_Curve_", safe_pathway, "_", cell_type, "_", genotype_label, ".pdf")
        full_out_path <- file.path(output_dir, fname)
        
        ggsave(full_out_path, p, width = 2.7, height = 2.7)
        message(paste0("    Saved Plot: ", full_out_path))
      }
    }
    
  }, error = function(e) {
    message(paste0("    Skipping ", cell_type, ": Error details: ", e$message))
  })
}

In [ ]:
# 1. Define your output directory (same as before)
dirs <- list()
dirs$gsea_pub <- paste0(load.dir, "/Publication_Figures_v1/10_GSEA_Publication_Style")

# 2. Load the Ravits dataset
ravits_df <- read.delim("./ravits_laser_capture.txt", sep = "\t", stringsAsFactors = FALSE)

# 3. Execute the function! 
# We'll call the genotype "sALS" (assuming Ravits is sporadic) and the cell type "LaserCapture"
run_gsea_pub_style(
  res_df = ravits_df, 
  genotype_label = "sALS", 
  cell_type = "LaserCapture", 
  output_dir = dirs$gsea_pub
)

In [ ]:
library(dplyr)
library(ggplot2)
library(tidyr)
library(stringr)

# =========================================================================
# FUNCTION 1: Load and format the data from the CSV directory
# =========================================================================
load_forest_data <- function(csv_dir, 
                             all_possible_cell_types = c("AlphaMN", "AlphaMN_nuclei", 
                                                         "GammaMN", "GammaMN_nuclei", 
                                                         "VisceralMN", "VisceralMN_nuclei", "LaserCapture"),
                             genotypes = c("SOD1", "C9orf72", "sALS", "PanALS"),
                             pathways = c("DM_Up", "DM_Down"),
                             pathway_labels = c(DM_Up = "DM Up",
                                                DM_Down = "DM Down")) {
  
  results_list <- list()
  missing_files <- c()
  
  for (ctype in all_possible_cell_types) {
    for (geno in genotypes) {
      expected_file <- file.path(csv_dir, paste0("Enrichment_Stats_", ctype, "_", geno, ".csv"))
      
      if (file.exists(expected_file)) {
        df <- read.csv(expected_file)
        required_cols <- c("pathway", "Wilcoxon_Pval", "Median_Shift", "ci_low", "ci_high")
        
        if (all(required_cols %in% colnames(df))) {
          for (path in pathways) {
            row_data <- df %>% filter(pathway == path)
            
            if (nrow(row_data) == 1) {
              results_list[[length(results_list) + 1]] <- data.frame(
                Cell_Type = ctype,
                Genotype = geno,
                Pathway = path,
                Median_Shift = row_data$Median_Shift,
                CI_Low = row_data$ci_low,
                CI_High = row_data$ci_high,
                P_Val = row_data$Wilcoxon_Pval
              )
            }
          }
        }
      } else {
        missing_files <- c(missing_files, expected_file)
      }
    }
  }
  
  if (length(missing_files) > 0) {
    message("--- Note: The following files were not found (Safe to ignore if intentional) ---")
    print(basename(missing_files))
  }
  
  if (length(results_list) == 0) {
    stop("No data loaded. Check directory and CI columns.")
  }
  
  plot_data <- bind_rows(results_list) %>%
    mutate(
      Pathway_Display = unname(pathway_labels[Pathway]),
      Pathway_Display = ifelse(is.na(Pathway_Display), Pathway, Pathway_Display),
      Label = paste0(Genotype, " \u00B7 ", Pathway_Display), 
      Significance = ifelse(P_Val < 0.05, "p < 0.05", "ns")
    )
  
  plot_data$Label <- factor(plot_data$Label, levels = rev(c(
    "SOD1 \u00B7 DM Up",    "SOD1 \u00B7 DM Down",
    "C9orf72 \u00B7 DM Up", "C9orf72 \u00B7 DM Down",
    "sALS \u00B7 DM Up",    "sALS \u00B7 DM Down",
    "PanALS \u00B7 DM Up",  "PanALS \u00B7 DM Down"
  )))
  
  return(plot_data)
}


# =========================================================================
# FUNCTION 2: Filter and generate the Unified Forest Plot
# =========================================================================

plot_unified_forest <- function(plot_data, target_cell_types, target_genotypes = NULL, 
                                output_dir, file_name = "Unified_ForestPlot.pdf", 
                                input_height = 5, input_width = 8) {
  
  # 1. Filter the data for requested cell types
  filtered_data <- plot_data %>% filter(Cell_Type %in% target_cell_types)
  
  # 2. Filter the data for requested genotypes (if provided)
  if (!is.null(target_genotypes)) {
    filtered_data <- filtered_data %>% filter(Genotype %in% target_genotypes)
  }
  
  # Safety check
  if (nrow(filtered_data) == 0) {
    stop("None of the requested target_cell_types / target_genotypes were found in the loaded data.")
  }
  
  # Adjust Wilcoxon p-values using FDR across the comparisons shown in this plot
  filtered_data <- filtered_data %>%
    mutate(
      P_Adj = p.adjust(P_Val, method = "BH"),
      Significance = ifelse(P_Adj < 0.05, "FDR < 0.05", "ns")
    )
  
  # 3. Lock the plotting order of the facets to exactly match the user's input list
  filtered_data$Cell_Type <- factor(filtered_data$Cell_Type, levels = target_cell_types)
  
  # 4. Generate the Plot
  p_forest <- ggplot(filtered_data, aes(x = Median_Shift, y = Label, color = Significance)) +
    
    facet_wrap(
      ~ Cell_Type,
      ncol = 1,
      strip.position = "right",
      labeller = labeller(Cell_Type = function(x) str_remove(x, "_nuclei$"))
    ) +
    geom_vline(xintercept = 0, color = "grey50", linewidth = 0.5) +
    geom_errorbarh(aes(xmin = CI_Low, xmax = CI_High), height = 0.2, linewidth = 0.8) +
    geom_point(size = 4, shape = 16) +
    
    scale_color_manual(
      values = c("FDR < 0.05" = "black", "ns" = "grey70"),
      breaks = c("FDR < 0.05", "ns")
    ) +
    
    labs(
      x = "Median Shift",
      y = NULL,
      color = "1-sided \nWilcoxon"
    ) +
    theme_classic(base_size = 20) +
    theme(
      plot.subtitle = element_text(hjust = 0.5, size = 20, color = "grey40", margin = margin(b = 15)),
      strip.background = element_rect(fill = "grey90", color = NA),
      strip.text = element_text(face = "bold", size = 13),
      panel.spacing = unit(1, "lines"),
      axis.text.y = element_text(color = "black", size = 20),
      axis.text.x = element_text(color = "black", size = 20),
      axis.line.y = element_line(linewidth = 0.8),
      axis.line.x = element_line(linewidth = 0.8),
      axis.ticks = element_line(linewidth = 0.8),
      legend.position = "right"
    )
  
  # 5. Save dynamically based on how many cell types were plotted
  out_path <- file.path(output_dir, file_name)
  
  ggsave(out_path, p_forest, width = input_width, height = input_height)
  message(paste0("Successfully saved plot to: ", out_path))
  
  return(p_forest)
}


In [ ]:

# --- 3. Run Loop ---
cell_types <- c("AlphaMN_nuclei", "GammaMN_nuclei", "VisceralMN_nuclei")

for (ctype in cell_types) {
  
  f_sod1 <- file.path(load.dir, paste0("GLMM_SC_DE_", ctype, "_SOD1_vs_Ctrl_AdjProp.csv"))
  if(file.exists(f_sod1)) {
    res <- read.csv(f_sod1, row.names = 1)
    run_gsea_pub_style(res, "SOD1", ctype, dirs$gsea_pub)
  }
  
  f_c9 <- file.path(load.dir, paste0("GLMM_SC_DE_", ctype, "_C9_vs_Ctrl_AdjProp.csv"))
  if(file.exists(f_c9)) {
    res <- read.csv(f_c9, row.names = 1)
    run_gsea_pub_style(res, "C9orf72", ctype, dirs$gsea_pub)
  }
  f_sals <- file.path(load.dir, paste0("GLMM_SC_DE_", ctype, "_SporadicALS_vs_Ctrl_AdjProp.csv"))
  if(file.exists(f_sals)) {
    res <- read.csv(f_sals, row.names = 1)
    run_gsea_pub_style(res, "sALS", ctype, dirs$gsea_pub)
  }
  f_panals <- file.path(load.dir, paste0("GLMM_SC_DE_", ctype, "_PanALS_vs_Ctrl_AdjProp.csv"))
  if(file.exists(f_panals)) {
    res <- read.csv(f_panals, row.names = 1)
    run_gsea_pub_style(res, "PanALS", ctype, dirs$gsea_pub)
  }
}

In [ ]:

# --- 3. Run Loop ---
cell_types <- c("AlphaMN", "GammaMN", "VisceralMN")

for (ctype in cell_types) {
  
  f_sod1 <- file.path(load.dir, paste0("GLMM_SC_DE_", ctype, "_SOD1_vs_Ctrl_AdjProp.csv"))
  if(file.exists(f_sod1)) {
    res <- read.csv(f_sod1, row.names = 1)
    run_gsea_pub_style(res, "SOD1", ctype, dirs$gsea_pub)
  }
  
  f_c9 <- file.path(load.dir, paste0("GLMM_SC_DE_", ctype, "_C9_vs_Ctrl_AdjProp.csv"))
  if(file.exists(f_c9)) {
    res <- read.csv(f_c9, row.names = 1)
    run_gsea_pub_style(res, "C9orf72", ctype, dirs$gsea_pub)
  }
  f_sals <- file.path(load.dir, paste0("GLMM_SC_DE_", ctype, "_SporadicALS_vs_Ctrl_AdjProp.csv"))
  if(file.exists(f_sals)) {
    res <- read.csv(f_sals, row.names = 1)
    run_gsea_pub_style(res, "sALS", ctype, dirs$gsea_pub)
  }
}

In [ ]:
csv_dir <- paste0(load.dir,"/Publication_Figures_v1/10_GSEA_Publication_Style")

# This loads everything it can find into one master dataframe
master_forest_data <- load_forest_data(csv_dir)

In [ ]:
plot_unified_forest(
  plot_data = master_forest_data, 
  target_cell_types = c("AlphaMN_nuclei", "GammaMN_nuclei"),
  output_dir = csv_dir,
  target_genotypes=c('SOD1', 'C9orf72', 'sALS'),
  file_name = "Forest_All_Nuclear.pdf"
)

In [ ]:
plot_unified_forest(
  plot_data = master_forest_data, 
  target_cell_types = c("LaserCapture"),
  output_dir = csv_dir,
  target_genotypes=c('SOD1', 'C9orf72', 'sALS'),
  file_name = "laser_capture_forest.pdf",
  input_width=10
)

In [ ]:
plot_unified_forest(
  plot_data = master_forest_data, 
  target_cell_types = c("AlphaMN", "GammaMN"),
  output_dir = csv_dir,
  file_name = "Forest_All_Fragments.pdf",
  input_height=9,
  input_width=9
)

## Figure 7F-G / Figure S12A-C: Fragment Subtypes

Annotate neuronal fragment classes, generate cluster and marker UMAPs, and create marker heatmaps for alphaFRAG, gammaFRAG, visceralFRAG, and interneuron-like fragment populations.


In [ ]:
# --- 0. Directory & Library Setup ---------------------------------------------
library(pheatmap)
library(Seurat)
library(ggplot2)
library(ggrastr)
library(patchwork)

dirs$sub_qc <- file.path(pub_dir, "0_QC_Cholinergic_Subtypes")
dir.create(dirs$sub_qc, recursive = TRUE, showWarnings = FALSE)

# --- 1. Define Cholinergic Subtype Markers ------------------------------------
alpha_markers    <- c("VIPR2", "PEX5L")
gamma_markers    <- c("AMTN", "IL33")
visceral_markers <- c("NOS1", "ZEB2")
other_markers    <- c("GAD1", "SLC6A5", "PRPH", "SLC17A6", "SLC5A7", "CHAT", "SLC6A1", "GAD2", "PITX2", "INA", "MAP2", "MAP1A", "NPR3") 

heatmap_features <- c(alpha_markers, gamma_markers, visceral_markers, other_markers)

# --- 2. Calculation: Average Expression & Scaling -----------------------------
avg_exp <- AverageExpression(axo_obj, features = heatmap_features, group.by = "seurat_clusters")$RNA
avg_exp_t <- t(avg_exp)

# Scale (Z-score) by Column (Gene)
mat_scaled <- apply(avg_exp_t, 2, function(x) {
  r <- range(x, na.rm = TRUE)
  if (diff(r) == 0) return(rep(0, length(x)))  # constant column
  (x - r[1]) / diff(r)
})
# --- 3. Visualization 1: Publication-Ready Heatmap ----------------------------
# Changes: 
# 1. Removed 'main' title
# 2. Adjusted fontsize_row/col to prevent smushing
# 3. Compacted the dendrogram trees

pdf_path <- file.path(dirs$sub_qc, "SubQC_Heatmap_Subtypes.pdf")
png_path <- file.path(dirs$sub_qc, "SubQC_Heatmap_Subtypes.png")

# Define common arguments for consistency
heatmap_args <- list(
  mat = mat_scaled,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "none",
  # color = colorRampPalette(c("navy", "white", "firebrick"))(100),
  border_color = "white",
  
  # Publication tweaks
  fontsize = 10,           # Base font size
  fontsize_row = 6,        # Slightly smaller rows to prevent overlap
  fontsize_col = 10,       # Column font size
  treeheight_row = 20,     # Compact row dendrogram
  treeheight_col = 20,     # Compact col dendrogram
  angle_col = 45,
  main = NA                # REMOVED TITLE
)

# Save PDF
do.call(pheatmap, c(heatmap_args, list(filename = pdf_path, width = 5, height = 4)))

# Save PNG
do.call(pheatmap, c(heatmap_args, list(filename = png_path, width = 5, height = 4)))

message(paste("Saved Clean Heatmap to:", dirs$sub_qc))

# --- 4. Visualization 2: Spatial Map (Cleaned) --------------------------------

# A. UMAP (Removed Title)
p_umap_sub <- DimPlot(axo_obj, reduction = "umap", label = TRUE, label.size = 4, raster = FALSE) + 
  theme_void() + 
  theme(legend.position = "none", plot.title = element_blank()) # No Title

# Rasterize points
p_umap_sub$layers[[1]] <- rasterise(p_umap_sub$layers[[1]], dpi = 300)

# B. Key Features
feats_to_map <- c("SLC5A7", "VIPR2", "AMTN", "NOS1", "GAD1", "SLC6A5")
plot_list <- list()

for(feat in feats_to_map) {
  # We keep the gene name (ggtitle) because otherwise the plot is unidentifiable,
  # but we can make it smaller/simpler if needed.
  p <- FeaturePlot(axo_obj, features = feat, order = TRUE, raster = FALSE) + 
    scale_color_viridis_c(option = "magma") + 
    ggtitle(feat) + # Gene name remains (essential context)
    theme_void() + 
    NoLegend() +
    theme(plot.title = element_text(size = 18, face = "italic", hjust = 0.5))
  
  p$layers[[1]] <- rasterise(p$layers[[1]], dpi = 300)
  plot_list[[feat]] <- p
}

# Layout
p_spatial_sub <- (p_umap_sub | plot_list$SLC5A7 | plot_list$VIPR2) / 
                 (plot_list$AMTN | plot_list$NOS1 | plot_spacer())

# Save using your existing helper or ggsave
if(exists("save_pub_plot")) {
  save_pub_plot(p_spatial_sub, "SubQC_Spatial_Subtypes", dirs$sub_qc, w=10, h=15)
} else {
  ggsave(file.path(dirs$sub_qc, "SubQC_Spatial_Subtypes.png"), p_spatial_sub, width=10, height=15, dpi=300)
}

print(p_spatial_sub)

In [ ]:
# --- 0. Directory & Library Setup ---------------------------------------------
library(Seurat)
library(ggplot2)
library(dplyr)
library(ggrastr) 
library(colorspace) 
library(ggrepel) 

if(!exists("pub_dir")) pub_dir <- "Plots" 
if(!exists("dirs")) dirs <- list()
dirs$mn_subtypes <- file.path(pub_dir, "MN_Subtypes_UMAP")
dir.create(dirs$mn_subtypes, recursive = TRUE, showWarnings = FALSE)

# --- 1. Define Metadata & Colors ----------------------------------------------
# 1. Create Metadata on the main object
# Initialize ALL cells as "Other" first
axo_obj$MN_Type <- "Other"

# Assign specific MN identities
axo_obj$MN_Type[axo_obj$seurat_clusters %in% c(0,4,20)]              <- "Alpha MNs"
axo_obj$MN_Type[axo_obj$seurat_clusters %in% c(5,30)]             <- "Gamma MNs"
axo_obj$MN_Type[axo_obj$seurat_clusters %in% c(3,11,14,16,32,33,37)]    <- "Visceral MNs"
axo_obj$MN_Type[axo_obj$seurat_clusters %in% c(1,2,6,7,10,24,29,38,39,40,47,48)] <- "Undetermined/Doublets"
axo_obj$MN_Type[axo_obj$seurat_clusters %in% c(17,9,34,23,36,12,28,46,45,41,18,43,15)] <- "Inhibitory INs"
axo_obj$MN_Type[axo_obj$seurat_clusters %in% c(42)]               <- "Cholinergic INs"
axo_obj$MN_Type[axo_obj$seurat_clusters %in% c(31,44,27,8,25,26,22,35,19,13,21)] <- "Excitatory INs"

# 2. Rename object to distinguish from nuclei data (as requested)
axo_debris <- axo_obj

# Set Factor Levels (Control plotting order: Other first so it stays in background)
axo_debris$MN_Type <- factor(axo_debris$MN_Type, 
                             levels = c("Other", 
                                        "Undetermined/Doublets",
                                        "Inhibitory INs",
                                        "Cholinergic INs",
                                        "Excitatory INs",
                                        "Alpha MNs", 
                                        "Gamma MNs", 
                                        "Visceral MNs"))
Idents(axo_debris) <- "MN_Type"

# 3. Define Palette with Grey for 'Other'
my_mn_colors <- c(
  "Alpha MNs"                = "#E41A1C", 
  "Gamma MNs"                = "#4DAF4A", 
  "Visceral MNs"             = "#377EB8",
  "Inhibitory INs"  = "#984EA3", # Purple
  "Cholinergic INs" = "#FF7F00", # Orange
  "Excitatory INs"  = "#F781BF", # Pink
  "Undetermined/Doublets"    = "#d9d9d9",  # Light grey for background clusters
  "Other"                    = "#d9d9d9"  # Light grey for background clusters
)

# 4. Define Darker Text Palette
text_colors <- darken(my_mn_colors, amount = 0.3)
names(text_colors) <- names(my_mn_colors)

# --- 2. Calculate Centroids ---------------------------------------------------
# Extract UMAP coordinates
umap_coords <- Embeddings(axo_debris, "umap") %>% as.data.frame()
colnames(umap_coords)[1:2] <- c("UMAP_1", "UMAP_2")
umap_coords$MN_Type <- axo_debris$MN_Type
umap_coords$seurat_clusters <- axo_debris$seurat_clusters # Needed for Plot 2

# Centroids for MN_Type (EXCLUDE "Other" from having a text label)
label_data_type <- umap_coords %>%
  filter(MN_Type != "Other") %>% 
  group_by(MN_Type) %>%
  summarise(UMAP_1 = median(UMAP_1), UMAP_2 = median(UMAP_2), .groups = 'drop')

# Centroids for specific Seurat clusters
label_data_cluster <- umap_coords %>%
  group_by(seurat_clusters) %>%
  summarise(UMAP_1 = median(UMAP_1), UMAP_2 = median(UMAP_2), .groups = 'drop')

# --- 3. Visualization 1: MN Types (Rasterized Points + Halo Labels) -----------
# Base Plot (Points Only)
p_umap_types_base <- DimPlot(axo_debris, 
                        reduction = "umap", 
                        group.by = "MN_Type",
                        cols = my_mn_colors,
                        pt.size = 1.5,
                        order = TRUE, # Brings colored cells to front
                        label = FALSE, 
                        raster = FALSE) + 
  theme_void() + 
  theme(legend.position = "none", plot.title = element_blank())

# Rasterize Points
if (length(p_umap_types_base$layers) > 0) {
  p_umap_types_base$layers[[1]] <- rasterise(p_umap_types_base$layers[[1]], dpi = 300)
}

# Add "Classy" Halo Labels
p_umap_types <- p_umap_types_base + 
  geom_text_repel(data = label_data_type, 
                  aes(x = UMAP_1, y = UMAP_2, label = MN_Type, color = MN_Type),
                  fontface = "bold", 
                  size = 6,
                  bg.color = "white", bg.r = 0.15,        
                  segment.color = NA, 
                  force = 5,          
                  seed = 42) +
  scale_color_manual(values = text_colors) 

# --- 4. Visualization 2: Raw Seurat Clusters ----------------------------------
# Create a new base plot grouped strictly by clusters using default Seurat colors
p_umap_clusters_base <- DimPlot(axo_debris, 
                        reduction = "umap", 
                        group.by = "seurat_clusters",
                        pt.size = 1.5,
                        label = FALSE, 
                        raster = FALSE) + 
  theme_void() + 
  theme(legend.position = "none", plot.title = element_blank())

# Rasterize Points
if (length(p_umap_clusters_base$layers) > 0) {
  p_umap_clusters_base$layers[[1]] <- rasterise(p_umap_clusters_base$layers[[1]], dpi = 300)
}

# Add anchored halo labels for cluster numbers
p_umap_clusters <- p_umap_clusters_base + 
  geom_text_repel(data = label_data_cluster, 
                  aes(x = UMAP_1, y = UMAP_2, label = seurat_clusters),
                  fontface = "bold", 
                  size = 5,                 
                  color = "black",         
                  bg.color = "white",        
                  bg.r = 0.15,              
                  segment.color = NA, 
                  force = 0, # Locks numbers directly onto the centroids             
                  seed = 42)

# --- 5. Save Outputs ----------------------------------------------------------
# Explicitly labeled as Debris
out_name_types    <- "MN_Subtypes_UMAP_Types_Debris"
out_name_clusters <- "MN_Subtypes_UMAP_Clusters_Debris"

if(exists("save_pub_plot")) {
  save_pub_plot(p_umap_types, out_name_types, dirs$mn_subtypes, w=6, h=6)
  save_pub_plot(p_umap_clusters, out_name_clusters, dirs$mn_subtypes, w=6, h=6)
} else {
  # Save Types Plot
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_types, ".pdf")), p_umap_types, width=8, height=6)
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_types, ".png")), p_umap_types, width=8, height=6, dpi=600)
  
  # Save Clusters Plot
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_clusters, ".pdf")), p_umap_clusters, width=8, height=6)
  ggsave(file.path(dirs$mn_subtypes, paste0(out_name_clusters, ".png")), p_umap_clusters, width=8, height=6, dpi=600)
}

message("Plots saved to: ", dirs$mn_subtypes)

In [ ]:
# --- 0. Directory & Library Setup ---------------------------------------------
library(pheatmap)
library(Seurat)
library(ggplot2)
library(ggrastr)
library(patchwork)

if(!exists("pub_dir")) pub_dir <- "Plots" 
if(!exists("dirs")) dirs <- list()
dirs$sub_qc <- file.path(pub_dir, "0_QC_Cholinergic_Subtypes")
dir.create(dirs$sub_qc, recursive = TRUE, showWarnings = FALSE)

# --- 1. Define Cholinergic Subtype Markers & Colors ---------------------------
alpha_markers    <- c("VIPR2", "PEX5L")
gamma_markers    <- c("AMTN", "IL33")
visceral_markers <- c("NOS1", "ZEB2")
other_markers    <- c("GAD1", "SLC6A5", "PRPH", "SLC17A6", "SLC5A7", "CHAT", "SLC6A1", "GAD2", "PITX2", "INA", "SLC6A5", "NPR3") 

heatmap_features <- c(alpha_markers, gamma_markers, visceral_markers, other_markers)

# Bring in your exact palette from the UMAP script
my_mn_colors <- c(
  "Alpha MNs"                = "#E41A1C", 
  "Gamma MNs"                = "#4DAF4A", 
  "Visceral MNs"             = "#377EB8",
  "Inhibitory INs"  = "#984EA3", 
  "Cholinergic INs" = "#FF7F00", 
  "Excitatory INs"  = "#F781BF"
  # Excluding Other and Doublets since they are filtered out below
)

# --- 2. Calculation: Average Expression & Scaling -----------------------------
# Exclude the unassigned/doublet cells before calculating averages
axo_heatmap_obj <- subset(axo_obj, subset = MN_Type != "Other" & MN_Type != "Undetermined/Doublets")

# Calculate averages using the new MN_Type labels
avg_exp <- AverageExpression(axo_heatmap_obj, features = heatmap_features, group.by = "MN_Type")$RNA
avg_exp_t <- t(avg_exp)

# Scale (0-1 normalization) by Column (Gene)
mat_scaled <- apply(avg_exp_t, 2, function(x) {
  r <- range(x, na.rm = TRUE)
  if (diff(r) == 0) return(rep(0, length(x)))  # constant column
  (x - r[1]) / diff(r)
})

# --- 2.5 Prepare Heatmap Annotations -------------------------------------------
# Create a dataframe mapping the rownames back to their cell type
row_anno <- data.frame(Cell_Type = rownames(mat_scaled))
rownames(row_anno) <- rownames(mat_scaled)

# Create a matching color list for pheatmap to read
anno_colors <- list(
  Cell_Type = my_mn_colors[rownames(mat_scaled)] # Strictly subsets to only the categories present
)

# --- 3. Visualization 1: Publication-Ready Heatmap ----------------------------
pdf_path <- file.path(dirs$sub_qc, "SubQC_Heatmap_Subtype_Rows.pdf")
png_path <- file.path(dirs$sub_qc, "SubQC_Heatmap_Subtype_Rows.png")

# Define common arguments for consistency
heatmap_args <- list(
  mat = mat_scaled,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  scale = "none",
  color = colorRampPalette(c("navy", "white", "firebrick"))(100),
  border_color = "white",
  annotation_legend=FALSE,
  
  # Inject the color annotations here
  annotation_row = row_anno,
  annotation_colors = anno_colors,
  
  # Publication tweaks
  fontsize = 8,           # Base font size
  fontsize_row = 12,        # Slightly smaller rows to prevent overlap
  fontsize_col = 8,       # Column font size
  treeheight_row = 20,     # Compact row dendrogram
  treeheight_col = 20,     # Compact col dendrogram
  angle_col = 45,
  main = NA                # REMOVED TITLE
)

# Save PDF
do.call(pheatmap::pheatmap, c(heatmap_args, list(filename = pdf_path, width = 4.3, height = 2.867)))

# Save PNG
do.call(pheatmap::pheatmap, c(heatmap_args, list(filename = png_path, width = 4.3, height = 2.867)))

message(paste("Saved Clean Heatmap to:", dirs$sub_qc))

In [ ]:
# --- 0. Directory & Library Setup ---------------------------------------------
library(Seurat)
library(ggplot2)
library(ggrastr)
library(patchwork)
library(dplyr)

dirs$sub_qc <- file.path(pub_dir, "0_QC_Cholinergic_Subtypes")
dir.create(dirs$sub_qc, recursive = TRUE, showWarnings = FALSE)

# --- 1. Visualization 1: Main Cluster UMAP (Saved Separately) -----------------
message("Generating Main UMAP...")

# Create base UMAP
p_umap_clusters <- DimPlot(axo_obj, reduction = "umap", label = TRUE, label.size = 5, raster = FALSE) + 
  theme_void() + 
  theme(legend.position = "none", plot.title = element_blank()) 

# Rasterize points at High Res (600 DPI)
# This keeps the vector text but makes the points a crisp bitmap
p_umap_clusters$layers[[1]] <- rasterise(p_umap_clusters$layers[[1]], dpi = 600)

# Save
ggsave(file.path(dirs$sub_qc, "SubQC_UMAP_Clusters.pdf"), p_umap_clusters, width = 5, height = 5)
ggsave(file.path(dirs$sub_qc, "SubQC_UMAP_Clusters.png"), p_umap_clusters, width = 5, height = 5, dpi = 600)

# --- 2. Visualization 2: Specific Marker Grid ---------------------------------
message("Generating Marker Grid...")

# Define markers
feats_to_map <- c("PRPH", "SLC17A6", "GAD1", "VIPR2", "AMTN", "NOS1")
plot_list <- list()

# Loop to create standardized plots
for(feat in feats_to_map) {
  p <- FeaturePlot(axo_obj, features = feat, order = TRUE, raster = FALSE, pt.size = 0.8) + 
    
    # Grey -> Red Gradient
    scale_color_gradient(low = "lightgrey", high = "#b30000") + 
    
    ggtitle(feat) + 
    theme_void() + 
    NoLegend() +
    theme(plot.title = element_text(size = 22, face = "italic", hjust = 0.5))
  
  # Rasterize points at High Res (600 DPI)
  p$layers[[1]] <- rasterise(p$layers[[1]], dpi = 600)
  
  plot_list[[feat]] <- p
}

# Arrange in a 2-row x 3-column grid
p_marker_grid <- wrap_plots(plot_list, ncol = 3, nrow = 2)

# Save Grid (Note: PDF will contain the 600 DPI raster layers inside)
ggsave(file.path(dirs$sub_qc, "SubQC_Spatial_Markers.pdf"), p_marker_grid, width = 12, height = 8)
ggsave(file.path(dirs$sub_qc, "SubQC_Spatial_Markers.png"), p_marker_grid, width = 12, height = 8, dpi = 600)

message(paste("Saved High-Res UMAP and Marker Grid to:", dirs$sub_qc))

# Print to screen
# print(p_marker_grid)

In [ ]:
# --- 1. Global Filters & Object Splitting -------------------------------------
axo_vh <- subset(axo_obj, subset = tissue == "VH")
# axo_vh <- axo_obj
Idents(axo_vh) <- "MN_Type"

target_objects <- list(
  "AlphaMN"    = subset(axo_vh, idents = "Alpha MNs"),
  "GammaMN"    = subset(axo_vh, idents = "Gamma MNs"),
  "VisceralMN" = subset(axo_vh, idents = "Visceral MNs"),
  "ExcitatoryIN" = subset(axo_vh, idents= "Excitatory INs"),
  "CholinergicIN" = subset(axo_vh, idents= "Cholinergic INs"),
  "InhibitoryIN" = subset(axo_vh, idents= "Inhibitory INs")
)


## Nebula Differential Expression For Figure Panels

Define and run single-cell GLMM differential expression for nuclei and fragment populations used by DAMNDM enrichment, TDP-43 heatmap, and alphaFRAG volcano panels.


In [ ]:
library(nebula)
library(Seurat)
library(dplyr)


run_sc_de_glmm <- function(input_obj, cell_type_name, output_dir, 
                           min_cells_input = 20, min_gene_pct = 0.1,
                           use_ncount = TRUE, use_mito = TRUE, use_batch = TRUE,
                           n_cores = 10) { 
  
  message(paste0("\n========================================================"))
  message(paste0(" Computing Single-Cell GLMM (NEBULA): ", cell_type_name))
  message(paste0("   > Cores: ", n_cores))
  message(paste0("========================================================"))
  input_obj$status[input_obj$status == "sALS"] <- "ALS"
  # A. Subset & Drop Levels (Original Subtype Logic)
  clean_obj <- subset(input_obj, subset = status %in% c("SOD1", "ctrl", "C9", "ALS"))
  clean_obj$status <- factor(clean_obj$status, levels = c("ctrl", "ALS", "C9", "SOD1"))
  
  # B. Filter Valid Samples
  sample_counts <- table(clean_obj$SampleID)
  valid_samples <- names(sample_counts[sample_counts >= min_cells_input])
  
  if (length(valid_samples) < 2) {
    message("  Skipping: Not enough valid samples.")
    return(NULL)
  }
  
  clean_obj <- subset(clean_obj, subset = SampleID %in% valid_samples)
  
  # Safety Checks
  clean_obj$status <- droplevels(clean_obj$status) 
  if (length(levels(clean_obj$status)) < 2) {
    message(paste0("  Skipping: Not enough disease groups left."))
    return(NULL)
  }
  if (!"ctrl" %in% levels(clean_obj$status)) {
    message("  Skipping: 'ctrl' reference group is missing.")
    return(NULL)
  }
  
  # C. Extract, SORT, and Filter Data
  meta_data <- clean_obj@meta.data
  
  # --- CHECK: Handle Missing nCount_RNA_noMT ---
  if (!"nCount_RNA_noMT" %in% colnames(meta_data)) {
    message("  Note: 'nCount_RNA_noMT' not found. Using 'nCount_RNA' for offset/covariate instead.")
    if ("nCount_RNA" %in% colnames(meta_data)) {
        meta_data$nCount_RNA_noMT <- meta_data$nCount_RNA
    } else {
        stop("Error: Neither 'nCount_RNA_noMT' nor 'nCount_RNA' found in metadata.")
    }
  }
  
  # Sort metadata (Required for Nebula)
  order_idx <- order(meta_data$SampleID)
  meta_data <- meta_data[order_idx, , drop = FALSE]
  
  # Extract counts
  cts <- GetAssayData(clean_obj, assay = "RNA", layer = "counts")
  cts <- cts[, rownames(meta_data), drop = FALSE]
  
  # Filter genes
  gene_detection_rate <- rowMeans(cts > 0)
  genes_to_keep <- names(gene_detection_rate[gene_detection_rate >= min_gene_pct])
  cts <- cts[genes_to_keep, , drop = FALSE]
  
  base_means <- rowMeans(cts)
  
  # D. Build Design Matrix (Subtypes)
  formula_terms <- c()
  
  if (use_batch) {
    meta_data$batch <- droplevels(as.factor(meta_data$batch))
    if (length(unique(meta_data$batch)) > 1) {
      formula_terms <- c(formula_terms, "batch")
    }
  }
  
  if (use_ncount) {
    if (sd(meta_data$nCount_RNA_noMT, na.rm = TRUE) > 0) {
      meta_data$nCount_scaled <- as.numeric(scale(meta_data$nCount_RNA_noMT))
      formula_terms <- c(formula_terms, "nCount_scaled")
    }
  }
  
  if (use_mito) {
    if (sd(meta_data$pct_counts_mito, na.rm = TRUE) > 0) {
      meta_data$mito_scaled <- as.numeric(scale(meta_data$pct_counts_mito))
      formula_terms <- c(formula_terms, "mito_scaled")
    }
  }
  
  formula_str <- paste("~ status", if(length(formula_terms) > 0) paste("+", paste(formula_terms, collapse = " + ")) else "")
  design_matrix <- model.matrix(as.formula(formula_str), data = meta_data)
  
  if (qr(design_matrix)$rank < ncol(design_matrix)) {
    message("  Warning: Matrix is rank-deficient. Falling back to ~ status.")
    design_matrix <- model.matrix(~ status, data = meta_data)
  }
  
  # E. Run NEBULA (Subtypes)
  subject_id <- as.character(meta_data$SampleID)
  message(paste0("  [1/2] Fitting Subtype model on ", n_cores, " cores..."))
  
  re <- nebula(count = cts, 
               id = subject_id, 
               pred = design_matrix, 
               offset = log(meta_data$nCount_RNA_noMT), 
               method = "LN",
               ncore = n_cores) 
  
  res_df <- re$summary
  
  # F. Export Results (Subtypes)
  save_glmm_res <- function(contrast_name, term_name, suffix_label) {
    
    logFC_col <- paste0("logFC_", term_name)
    se_col    <- paste0("se_", term_name)
    pval_col  <- paste0("p_", term_name)
    
    if (logFC_col %in% colnames(res_df)) {
      
      out_df <- data.frame(
        row.names      = res_df$gene,
        baseMean       = base_means[res_df$gene],
        log2FoldChange = res_df[[logFC_col]] / log(2),      
        lfcSE          = res_df[[se_col]] / log(2),          
        pvalue         = res_df[[pval_col]]
      )
      
      out_df$stat <- out_df$log2FoldChange / out_df$lfcSE
      out_df$padj <- p.adjust(out_df$pvalue, method = "BH")
      
      out_df <- out_df %>% arrange(padj)
      
      filename <- file.path(output_dir, paste0("GLMM_SC_DE_", cell_type_name, "_", suffix_label, ".csv"))
      
      write.csv(out_df, filename, row.names = TRUE)
      message(paste0("    Saved: ", filename))
    }
  }
  
  save_glmm_res("SporadicALS", "statusALS", "SporadicALS_vs_Ctrl_AdjProp")
  save_glmm_res("C9", "statusC9", "C9_vs_Ctrl_AdjProp")
  save_glmm_res("SOD1", "statusSOD1", "SOD1_vs_Ctrl_AdjProp")
  
  # --- G. Run NEBULA (Pan-ALS) --------------------------------------------
  message(paste0("  [2/2] Fitting Pan-ALS model (Binary)..."))
  
  # 1. Create Binary Metadata (All ALS vs Ctrl)
  meta_pan <- meta_data
  meta_pan$status <- as.character(meta_pan$status)
  meta_pan$status[meta_pan$status != "ctrl"] <- "ALS" # Pool everything else into ALS
  meta_pan$status <- factor(meta_pan$status, levels = c("ctrl", "ALS"))
  
  # 2. Re-build Design Matrix for Binary Status
  # We reuse 'formula_str' logic but with the new binary 'status'
  design_matrix_pan <- model.matrix(as.formula(formula_str), data = meta_pan)
  
  if (qr(design_matrix_pan)$rank < ncol(design_matrix_pan)) {
      design_matrix_pan <- model.matrix(~ status, data = meta_pan)
  }

  # 3. Run NEBULA Again
  re_pan <- nebula(count = cts, 
                   id = subject_id, 
                   pred = design_matrix_pan, 
                   offset = log(meta_data$nCount_RNA_noMT), 
                   method = "LN",
                   ncore = n_cores)
  
  # 4. Save Pan-ALS Result
  # We overwrite res_df so we can reuse the helper function
  res_df <- re_pan$summary
  
  # In this model, 'statusALS' refers to the pooled ALS group
  save_glmm_res("PanALS", "statusALS", "PanALS_vs_Ctrl_AdjProp")

  message("Analysis and exports complete.")
}

In [ ]:
run_sc_de_glmm(
    input_obj = target_objects_nuc$AlphaMN_nuclei, 
    cell_type_name = 'AlphaMN_nuclei', 
    output_dir = load.dir, 
    min_cells_input = 0, 
    min_gene_pct = 0.05,
    use_ncount = FALSE, 
    use_mito = FALSE,
    use_batch = TRUE,
    n_cores=1 
    # use_ashr=TRUE
  )

In [ ]:
run_sc_de_glmm(
    input_obj = target_objects_nuc$GammaMN_nuclei, 
    cell_type_name = 'GammaMN_nuclei', 
    output_dir = load.dir, 
    min_cells_input = 0, 
    min_gene_pct = 0.05,
    use_ncount = FALSE, 
    use_mito = FALSE,
    use_batch = TRUE,
    n_cores=1 
    # use_ashr=TRUE
  )

In [ ]:
run_sc_de_glmm(
    input_obj = target_objects_nuc$VisceralMN_nuclei, 
    cell_type_name = 'VisceralMN_nuclei', 
    output_dir = load.dir, 
    min_cells_input = 0, 
    min_gene_pct = 0.05,
    use_ncount = FALSE, 
    use_mito = FALSE,
    use_batch = TRUE,
    n_cores=1 
    # use_ashr=TRUE
  )

In [ ]:
run_sc_de_glmm(
    input_obj = target_objects$AlphaMN, 
    cell_type_name = 'AlphaMN', 
    output_dir = load.dir, 
    min_cells_input = 5, 
    min_gene_pct = 0.01,
    use_ncount = TRUE, 
    use_mito = TRUE,
    use_batch = TRUE,
    n_cores=1
  )

In [ ]:
run_sc_de_glmm(
    input_obj = target_objects$ExcitatoryIN, 
    cell_type_name = 'ExcitatoryIN', 
    output_dir = load.dir, 
    min_cells_input = 5, 
    min_gene_pct = 0.01,
    use_ncount = TRUE, 
    use_mito = TRUE,
    use_batch = TRUE,
    n_cores=1
  )

In [ ]:
run_sc_de_glmm(
    input_obj = target_objects$InhibitoryIN, 
    cell_type_name = 'InhibitoryIN', 
    output_dir = load.dir, 
    min_cells_input = 5, 
    min_gene_pct = 0.01,
    use_ncount = TRUE, 
    use_mito = TRUE,
    use_batch = TRUE,
    n_cores=1
  )

In [ ]:
run_sc_de_glmm(
    input_obj = target_objects$GammaMN, 
    cell_type_name = 'GammaMN', 
    output_dir = load.dir, 
    min_cells_input = 5, 
    min_gene_pct = 0.01,
    use_ncount = TRUE, 
    use_mito = TRUE,
    use_batch = TRUE,
    n_cores=1
  )

In [ ]:
run_sc_de_glmm(
    input_obj = target_objects$VisceralMN, 
    cell_type_name = 'VisceralMN', 
    output_dir = load.dir, 
    min_cells_input = 5, 
    min_gene_pct = 0.01,
    use_ncount = TRUE, 
    use_mito = TRUE,
    use_batch = TRUE,
    n_cores=1
  )

## Figure 7H / Figure S12G-J: TDP-43 Heatmap And AlphaFRAG Volcanoes

Generate alphaFRAG Nebula volcano plots and the TDP-43 cryptic exon target heatmap across neuronal subtype contrasts.


In [ ]:
library(ggplot2)
library(ggrepel)
library(dplyr)
library(tibble)

# --- 1. Define Reusable Volcano Function (With Custom Annotations) ------------
make_nebula_volcano <- function(res_df, title, filename_base, output_dir, 
                                lfc_cap = 10,
                                filter_unannotated = TRUE,
                                custom_genes = NULL,
                                n_top=5) { 
  
  message(paste0("Generating Volcano: ", title))
  
  # A. Data Cleanup & Prep
  df_clean <- as.data.frame(res_df) 
  if(!"gene" %in% colnames(df_clean)) {
    df_clean <- rownames_to_column(df_clean, var = "gene")
  }
  
  # Filter out NAs
  df_clean <- df_clean %>% filter(!is.na(padj) & !is.na(log2FoldChange))
  
  # --- Filter Unannotated ---
  if (filter_unannotated) {
    df_clean <- df_clean %>% 
      filter(!grepl("^(AC|AL|AP|AF|BX|CR|CU)[0-9]+\\.", gene))
  }

  # --- Outlier Clamping ---
  df_clean$is_clamped <- "No"
  
  upper_outliers <- df_clean$log2FoldChange > lfc_cap
  if(any(upper_outliers)) {
    df_clean$log2FoldChange[upper_outliers] <- lfc_cap
    df_clean$is_clamped[upper_outliers] <- "Yes"
  }
  
  lower_outliers <- df_clean$log2FoldChange < -lfc_cap
  if(any(lower_outliers)) {
    df_clean$log2FoldChange[lower_outliers] <- -lfc_cap
    df_clean$is_clamped[lower_outliers] <- "Yes"
  }
  
  # B. Significance Classes
  df_clean$diffexpressed <- "NO"
  df_clean$diffexpressed[df_clean$log2FoldChange > 0.25 & df_clean$padj < 0.1] <- "UP"
  df_clean$diffexpressed[df_clean$log2FoldChange < -0.25 & df_clean$padj < 0.1] <- "DOWN"
  
  # C. Labeling Strategy (Prioritize Custom Genes ONLY if Significant)
  df_clean$delabel <- NA
  
  # 1. Identify Custom Genes present in data AND Significant
  custom_found <- character(0)
  if (!is.null(custom_genes)) {
    # Ensure we only pick custom genes that are actually UP or DOWN
    custom_found <- df_clean %>% 
      filter(gene %in% custom_genes & diffexpressed != "NO") %>% 
      pull(gene)
  }
  
  # 2. Identify Top Genes (excluding custom ones to avoid double counting)
  sig_up <- df_clean %>% filter(diffexpressed == "UP", !gene %in% custom_found)
  sig_dn <- df_clean %>% filter(diffexpressed == "DOWN", !gene %in% custom_found)
  
  # Pick top remaining UP and DOWN
  top_up <- if(nrow(sig_up) > 0) sig_up %>% arrange(padj) %>% head(n_top) %>% pull(gene) else character(0)
  top_dn <- if(nrow(sig_dn) > 0) sig_dn %>% arrange(padj) %>% head(n_top) %>% pull(gene) else character(0)
  
  # 3. Combine Lists (Custom first)
  genes_to_label <- c(custom_found, top_up, top_dn)
  
  # Apply labels
  df_clean$delabel[df_clean$gene %in% genes_to_label] <- df_clean$gene[df_clean$gene %in% genes_to_label]
  
  # D. Visualization
  p_volcano <- ggplot(df_clean, aes(x = log2FoldChange, y = -log10(padj), 
                                    col = diffexpressed, label = delabel)) +
    
    geom_point(aes(shape = is_clamped), alpha = 0.6, size = 1.5) +
    
    scale_color_manual(values = c("DOWN" = "steelblue", "NO" = "grey85", "UP" = "firebrick")) +
    scale_shape_manual(values = c("No" = 16, "Yes" = 17)) + 
    
    xlim(-lfc_cap - 1, lfc_cap + 1) + 
    
    # --- SPLIT LABELS TO PREVENT CROSSING Y-AXIS ---
    
    # 1. Labels for DOWN genes (Left side)
    geom_text_repel(
      data = df_clean %>% filter(log2FoldChange < 0 & !is.na(delabel)),
      xlim = c(NA, -0.2), 
      max.overlaps = 50, 
      box.padding = 0.5, 
      point.padding = 0.3,
      size = 4,
      fontface = "italic",
      segment.color = "grey50",
      force = 2,
      # --- HALO EFFECT ---
      bg.color = "white",
      bg.r = 0.15
    ) +
    
    # 2. Labels for UP genes (Right side)
    geom_text_repel(
      data = df_clean %>% filter(log2FoldChange > 0 & !is.na(delabel)),
      xlim = c(0.2, NA), 
      max.overlaps = 50, 
      box.padding = 0.5, 
      point.padding = 0.3,
      size = 4,
      fontface = "italic",
      segment.color = "grey50",
      force = 2,
      # --- HALO EFFECT ---
      bg.color = "white",
      bg.r = 0.15
    ) +
    
    theme_classic(base_size = 22) +
    labs(
      title = title, 
      subtitle = paste0(nrow(filter(df_clean, diffexpressed=="UP")), " UP | ", 
                        nrow(filter(df_clean, diffexpressed=="DOWN")), " DOWN"),
      x = "Log2 Fold Change", 
      y = "-Log10 Adjusted P-value"
    ) +
    theme(
      legend.position = "none",
      plot.title = element_text(hjust = 0.5),
      plot.subtitle = element_text(hjust = 0.5, color = "grey30")
    )
  
  # E. Save
  out_pdf <- file.path(output_dir, paste0(filename_base, ".pdf"))
  out_png <- file.path(output_dir, paste0(filename_base, ".png"))
  
  ggsave(out_pdf, p_volcano, width = 6, height = 5)
  ggsave(out_png, p_volcano, width = 6, height = 5, dpi = 600)
  
  message(paste("Saved:", out_png))
}

# --- 2. Execution Loop --------------------------------------------------------

cell_types <- c("AlphaMN", "GammaMN", "VisceralMN", "InhibitoryIN", "ExcitatoryIN", "CholinergicIN") 
contrasts  <- c("SOD1", "C9", "SporadicALS", "PanALS")

# Define genes you ALWAYS want labeled (if present AND significant)
my_custom_list <- c("DDIT3", "GAP43", "ATF3", "UNC13A", "SYT4", "AGTPBP1")

target_folder <- file.path(load.dir, "Publication_Figures_v1", "Volcano_Plots", "Debris")
if (!dir.exists(target_folder)) dir.create(target_folder, recursive = TRUE)

for (ctype in cell_types) {
  for (cond in contrasts) {
    f_name <- file.path(load.dir, paste0("GLMM_SC_DE_", ctype, "_", cond, "_vs_Ctrl_AdjProp.csv"))
    
    if (file.exists(f_name)) {
      res_df <- read.csv(f_name, row.names = 1) 
      
      make_nebula_volcano(
        res_df        = res_df, 
        title         = paste0(ctype, ": ", cond, " vs. Ctrl"), 
        filename_base = paste0("Volcano_Nebula_", ctype, "_", cond), 
        output_dir    = target_folder,
        lfc_cap       = 4,
        filter_unannotated = TRUE,
        n_top = 10,
        custom_genes  = my_custom_list 
      )
    }
  }
}

In [ ]:
library(pheatmap)
library(dplyr)
library(tidyr)
library(tibble)
library(RColorBrewer)
library(grid) 

# --- UPDATED TARGET LIST ---
tdp43_targets <- c(
  "STMN2", "UNC13A", "CYFIP2", "HDGFL2", "AGRN", "PFKP", "SYT7",
  "KALRN", "ATG4B", "ARHGAP32", "ELAVL3", "PRUNE2", "NUP188",
  "CAMK2B", "CDK7", "ATP8A2"
)

run_tdp43_target_heatmap <- function(result_dir, 
                                     cell_types = c("AlphaMN", "GammaMN"),
                                     contrasts = c("SOD1", "C9", "SporadicALS"),
                                     target_genes = tdp43_targets,
                                     sig_threshold_color = 0.1,    
                                     cluster_all_cols = FALSE,
                                     group_by_status = TRUE,
                                     status_labels = c(SOD1 = "SOD1",
                                                       C9 = "C9orf72",
                                                       SporadicALS = "sALS")) { 
  
  message(paste0("Generating TDP-43 Heatmap (Group by Status: ", group_by_status, ")..."))
  
  # --- 1. Load Data ---
  long_df_list <- list()
  
  for (ct in cell_types) {
    for (cond in contrasts) {
      f_name <- file.path(result_dir, paste0("GLMM_SC_DE_", ct, "_", cond, "_vs_Ctrl_AdjProp.csv"))
      
      if (file.exists(f_name)) {
        tmp <- read.csv(f_name)
        if (!"gene" %in% colnames(tmp)) colnames(tmp)[1] <- "gene"
        
        tmp_subset <- tmp %>% filter(gene %in% target_genes)
        
        if (nrow(tmp_subset) > 0) {
          tmp_subset$padj_subset <- p.adjust(tmp_subset$pvalue, method = "BH")
          tmp_subset$cell_type <- ct
          tmp_subset$condition <- cond
          tmp_subset$group_id  <- paste(ct, cond, sep = "_") 
          long_df_list[[paste(ct, cond)]] <- tmp_subset
        }
      }
    }
  }
  
  if (length(long_df_list) == 0) {
    warning("No target genes found.")
    return(NULL)
  }

  plot_data <- bind_rows(long_df_list)
  
  # --- 2. Construct Matrices ---
  mat_lfc <- plot_data %>%
    select(gene, group_id, log2FoldChange) %>%
    pivot_wider(names_from = group_id, values_from = log2FoldChange, values_fill = 0) %>%
    column_to_rownames("gene") %>%
    as.matrix()
  
  mat_padj <- plot_data %>%
    select(gene, group_id, padj_subset) %>%
    pivot_wider(names_from = group_id, values_from = padj_subset, values_fill = 1) %>%
    column_to_rownames("gene") %>%
    as.matrix()
  
  common_genes <- intersect(rownames(mat_lfc), rownames(mat_padj))
  mat_lfc <- mat_lfc[common_genes, , drop = FALSE]
  mat_padj <- mat_padj[common_genes, , drop = FALSE]
  
  # --- 3. Column Ordering Logic ---
  gaps_at <- NULL
  ordered_cols <- c()
  
  if (!cluster_all_cols) {
    gaps_at <- c()
    running_total <- 0
    
    if (group_by_status) {
      for (cond in contrasts) {
        current_block_cols <- c()
        for (ct in cell_types) {
          target_col <- paste(ct, cond, sep = "_")
          if (target_col %in% colnames(mat_lfc)) {
            current_block_cols <- c(current_block_cols, target_col)
          }
        }
        if (length(current_block_cols) > 0) {
          ordered_cols <- c(ordered_cols, current_block_cols)
          running_total <- running_total + length(current_block_cols)
          gaps_at <- c(gaps_at, running_total)
        }
      }
    } else {
      for (ct in cell_types) {
        current_block_cols <- c()
        for (cond in contrasts) {
          target_col <- paste(ct, cond, sep = "_")
          if (target_col %in% colnames(mat_lfc)) {
            current_block_cols <- c(current_block_cols, target_col)
          }
        }
        if (length(current_block_cols) > 0) {
          ordered_cols <- c(ordered_cols, current_block_cols)
          running_total <- running_total + length(current_block_cols)
          gaps_at <- c(gaps_at, running_total)
        }
      }
    }

    mat_lfc <- mat_lfc[, ordered_cols, drop = FALSE]
    mat_padj <- mat_padj[, ordered_cols, drop = FALSE]

    if (length(gaps_at) > 0) {
      gaps_at <- gaps_at[-length(gaps_at)]
    }
  }
  
  # --- 4. Display Settings & Row Filtering ---
  display_mat <- matrix("", nrow = nrow(mat_padj), ncol = ncol(mat_padj))
  rownames(display_mat) <- rownames(mat_padj)
  colnames(display_mat) <- colnames(mat_padj)
  
  display_mat[mat_padj < 0.1] <- "*"
  display_mat[mat_padj < 0.05] <- "**"
  display_mat[mat_padj < 0.01] <- "***"
  
  mat_plot <- mat_lfc
  mat_plot[mat_padj > sig_threshold_color] <- 0
  
  rows_to_keep <- rowSums(abs(mat_plot)) > 0
  
  if (sum(rows_to_keep) == 0) {
    warning("No significant genes remaining after filtering. Skipping plot.")
    return(NULL)
  }
  
  mat_plot <- mat_plot[rows_to_keep, , drop = FALSE]
  display_mat <- display_mat[rows_to_keep, , drop = FALSE]
  
  # --- 5. Annotations with Display Label Mapping ---
  col_cell_types <- gsub("_.*", "", colnames(mat_plot))
  col_status     <- gsub("^[^_]+_", "", colnames(mat_plot))

  col_status_display <- unname(status_labels[col_status])
  col_status_display[is.na(col_status_display)] <- col_status[is.na(col_status_display)]

  status_levels_display <- unname(status_labels[contrasts])
  status_levels_display[is.na(status_levels_display)] <- contrasts[is.na(status_levels_display)]
  
  annotation_col <- data.frame(
    Status   = factor(col_status_display, levels = status_levels_display),
    CellType = factor(col_cell_types, levels = intersect(cell_types, col_cell_types))
  )
  rownames(annotation_col) <- colnames(mat_plot)
  
  status_colors_internal <- c(SOD1 = "#56B4E9", C9 = "#E69F00", SporadicALS = "#009E73")
  status_colors_display <- status_colors_internal[contrasts]
  names(status_colors_display) <- status_levels_display
  status_colors_display <- status_colors_display[!is.na(status_colors_display)]

  ann_colors <- list(
    CellType = c(AlphaMN = "grey90", GammaMN = "grey40", VisceralMN = "black",
                 InhibitoryIN = "#CC79A7", ExcitatoryIN = "#D55E00", CholinergicIN = "#F0E442"), 
    Status = status_colors_display
  )

  ann_colors$CellType <- ann_colors$CellType[names(ann_colors$CellType) %in% unique(col_cell_types)]
  
  clean_colnames <- paste(col_cell_types, col_status_display)
  
  # --- 6. Draw ---
  my_colors <- colorRampPalette(c("navy", "white", "firebrick"))(100)
  my_breaks <- seq(-2, 2, length.out = 101)
  
  hm_obj <- pheatmap(mat_plot,
                     color = my_colors,
                     breaks = my_breaks,
                     cluster_rows = TRUE, 
                     cluster_cols = cluster_all_cols, 
                     gaps_col = gaps_at,
                     annotation_col = annotation_col,
                     annotation_colors = ann_colors,
                     labels_col = clean_colnames,
                     fontsize_row = 12,
                     fontsize_col = 10, 
                     display_numbers = display_mat,
                     number_color = "black", 
                     fontsize = 10,
                     show_colnames = FALSE,
                     silent = TRUE) 
  
  group_label <- if (group_by_status) "ByStatus" else "ByCellType"
  suffix <- if (cluster_all_cols) "Clustered" else group_label
  
  out_file <- file.path(result_dir, paste0("TDP43_Targets_Heatmap_", suffix, ".pdf"))
  
  h_calc <- max(3, nrow(mat_plot) * 0.3)
  
  pdf(out_file, width = 6, height = h_calc)
  grid::grid.draw(hm_obj$gtable) 
  dev.off()
  
  message(paste("Saved TDP-43 heatmap to:", out_file))
}



run_tdp43_target_heatmap(load.dir, 
                   cell_types = c("AlphaMN", "GammaMN", "VisceralMN", "ExcitatoryIN", "InhibitoryIN"), 
                         contrasts = c("SOD1", "C9", "SporadicALS"),
                         sig_threshold_color = 0.1, 
                         cluster_all_cols = TRUE,
                         group_by_status = FALSE)

run_tdp43_target_heatmap(load.dir, 
                   cell_types = c("AlphaMN", "GammaMN", "VisceralMN", "ExcitatoryIN", "InhibitoryIN"), 
                         contrasts = c("SOD1", "C9", "SporadicALS"),
                         sig_threshold_color = 0.1, 
                         cluster_all_cols = FALSE,
                         group_by_status = FALSE)

## Assay State For AlphaFRAG Scoring

Set the AlphaMN fragment assay before downstream DAMNDM scoring. This is retained as a state-setting cell even though it is not itself a figure panel.


In [ ]:
DefaultAssay(target_objects$AlphaMN) <- 'RNA_noMT'

## Figure 7D-E/J-K / Figure S11C-E / Figure S12E-F: DAMNDM Scores And LMMs

Run the DAMNDM weighted-score pipeline for nuclei and fragment subtypes, including density plots, sample/donor means, and sample-level random-effect LMM statistics.


In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(patchwork)
library(tibble)
library(lme4)
library(lmerTest) 
library(emmeans)  

# --- 0. Check for Homologene --------------------------------------------------
if (!require("homologene", quietly = TRUE)) {
  stop("Please install homologene: install.packages('homologene')")
}

# --- HELPER FUNCTION: Calculate WEIGHTED Z-Scores -----------------------------
calc_weighted_score <- function(genes, sig_table, data_matrix) {
  # 1. Get weights (Abs Log2FC)
  w_table <- sig_table %>% 
    filter(human_gene %in% genes) %>%
    dplyr::select(human_gene, log2FoldChange) %>% 
    distinct(human_gene, .keep_all=TRUE)
  
  # 2. Align weights to the data matrix rows
  valid_genes <- intersect(rownames(data_matrix), w_table$human_gene)
  
  if(length(valid_genes) == 0) return(rep(0, ncol(data_matrix)))

  w_vector <- abs(w_table$log2FoldChange[match(valid_genes, w_table$human_gene)])
  
  # 3. Subset matrix
  sub_mat <- data_matrix[valid_genes, , drop=FALSE]
  
  # 4. Compute Weighted Average
  weighted_sums <- colSums(sub_mat * w_vector, na.rm=TRUE)
  final_scores  <- weighted_sums / sum(w_vector)
  
  return(final_scores)
}

run_damn_pipeline <- function(
    seurat_obj, 
    celltype_prefix,
    conditions_to_include = c("SOD1", "ctrl", "C9", "ALS", "sALS"),
    pvalue_threshold = 0.05,
    top_n_genes = 500,
    save_dir_dist = "plots/1_Distributions",
    save_dir_lmm = "plots/3_LMM_Stats",
    signature_csv_path = "damn_signature.csv",
    obj_type = "n Cells"
) {
  
  message(paste0("\n======================================================="))
  message(paste0("Running Pipeline for: ", celltype_prefix))
  message(paste0("=======================================================\n"))
  
  dir.create(save_dir_dist, recursive = TRUE, showWarnings = FALSE)
  dir.create(save_dir_lmm, recursive = TRUE, showWarnings = FALSE)
  
  # --- 1. Prepare Object ---
  merged_obj <- subset(seurat_obj, subset = status %in% conditions_to_include)
    
  current_status <- as.character(merged_obj$status)
  genotype_vec <- dplyr::recode(current_status,
    "ctrl" = "Control",
    "C9"   = "C9orf72",
    "ALS"  = "sALS",
    "sALS" = "sALS",
    "SOD1" = "SOD1"
  )

  merged_obj$Status <- factor(genotype_vec, levels = c("Control", "SOD1", "C9orf72", "sALS"))

  message("Object prepared. Status counts:")
  print(table(merged_obj$Status))

  # --- 2. Load Mouse Signature ---
  mouse_damn_raw <- read.csv(signature_csv_path, stringsAsFactors = FALSE)

  if (!"gene" %in% colnames(mouse_damn_raw)) colnames(mouse_damn_raw)[1] <- "gene"
  if (!"log2FoldChange" %in% colnames(mouse_damn_raw)) stop("CSV must have 'log2FoldChange' column")
  if (!"padj" %in% colnames(mouse_damn_raw)) stop("CSV must have 'padj' column")

  mouse_damn_raw <- mouse_damn_raw %>%
    arrange(padj) %>%
    distinct(gene, .keep_all = TRUE)

  message("Loaded ", nrow(mouse_damn_raw), " unique mouse DM genes.")

  # --- 3. Map Mouse -> Human Orthologs ---
  hg_map <- homologene::homologene(unique(mouse_damn_raw$gene), inTax = 10090, outTax = 9606)

  ortholog_map <- hg_map %>%
    as_tibble() %>%
    dplyr::rename(mouse_gene = `10090`, human_gene = `9606`) %>%
    filter(!is.na(human_gene), human_gene != "")

  damn_sig_mapped <- mouse_damn_raw %>%
    inner_join(ortholog_map, by = c("gene" = "mouse_gene"))

  # --- 4. Filter for Detectability ---
  total_cells <- ncol(merged_obj)
  counts_mat <- GetAssayData(merged_obj, layer = "counts")
  gene_counts <- Matrix::rowSums(counts_mat > 0)
  detectable_genes <- names(gene_counts)[(gene_counts / total_cells) > 0.01]

  damn_sig_detectable <- damn_sig_mapped %>%
    filter(human_gene %in% detectable_genes)

  # --- 5. Define Gene Panels ---
  damn_injury_genes <- damn_sig_detectable %>%
    filter(padj < pvalue_threshold, log2FoldChange > 0) %>%
    arrange(desc(log2FoldChange)) %>%
    head(top_n_genes) %>%
    pull(human_gene)

  damn_homeo_genes <- damn_sig_detectable %>%
    filter(padj < pvalue_threshold, log2FoldChange < 0) %>%
    arrange(log2FoldChange) %>%
    head(top_n_genes) %>%
    pull(human_gene)

  final_damn_injury <- unique(damn_injury_genes)
  final_damn_homeo  <- unique(damn_homeo_genes)

  message("Final Panel: ", length(final_damn_injury), " Injury markers / ", length(final_damn_homeo), " Homeostatic markers.")

  # --- 6. Calculate Weighted Z-Scores ---
  merged_obj <- ScaleData(merged_obj, features = c(final_damn_injury, final_damn_homeo), verbose = FALSE)
  scale_mat <- GetAssayData(merged_obj, layer = "scale.data")

  message("Calculating Weighted Z-Scores (Weight = |Log2FC|)...")
  
  # Keep DAMN in internal metadata columns for downstream compatibility.
  merged_obj$Score_DAMN_Injury <- calc_weighted_score(final_damn_injury, damn_sig_detectable, scale_mat)
  merged_obj$Score_DAMN_Homeo  <- calc_weighted_score(final_damn_homeo, damn_sig_detectable, scale_mat)
  merged_obj$Score_DAMN_Delta  <- merged_obj$Score_DAMN_Injury - merged_obj$Score_DAMN_Homeo

  # --- 7. Principled Thresholding & Classification ---
  plot_df <- data.frame(
    Status = merged_obj$Status,
    Injury_Z = merged_obj$Score_DAMN_Injury,
    Homeo_Z  = merged_obj$Score_DAMN_Homeo,
    Delta    = merged_obj$Score_DAMN_Delta
  )

  control_deltas <- plot_df %>% filter(Status == "Control") %>% pull(Delta)
  DAMN_THRESHOLD <- quantile(control_deltas, 0.95)

  message("Principled Threshold (95th % of Control): ", round(DAMN_THRESHOLD, 3))

  merged_obj$mouse_DAMN_State <- ifelse(merged_obj$Score_DAMN_Delta > DAMN_THRESHOLD, "Injured", "Homeostatic")
  plot_df$mouse_DAMN_State <- merged_obj$mouse_DAMN_State 

  # --- 8. Visualization ---
  my_colors <- c("Control" = "grey70", "SOD1" = "firebrick", "C9orf72" = "orange", "sALS" = "steelblue")

  p_scatter <- ggplot(plot_df, aes(x = Homeo_Z, y = Injury_Z, color = Status)) +
    geom_point(alpha = 0.4, size = 0.5) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "black") + 
    scale_color_manual(values = my_colors) +
    theme_classic() +
    ggtitle(paste0(celltype_prefix, ": DM vs. Homeostatic")) +
    xlab("Homeostatic Score (Weighted Z)") +
    ylab("DM/Injury Score (Weighted Z)")

  p_dens <- ggplot(plot_df, aes(x = Delta, fill = Status)) +
    geom_density(alpha = 0.5) +
    scale_fill_manual(values = my_colors) +
    theme_classic(base_size = 22) +
    xlab("Weighted DM Score") + 
    ylab("Density")

  stats_damn <- plot_df %>%
    group_by(Status, mouse_DAMN_State) %>%
    summarise(Count = n(), .groups = "drop") %>%
    mutate(Total = sum(Count), .by = Status) %>%
    mutate(Percentage = Count / Total * 100) %>%
    filter(mouse_DAMN_State == "Injured")

  p_bar <- ggplot(stats_damn, aes(x = Status, y = Percentage, fill = Status)) +
    geom_bar(stat = "identity", color = "black", width = 0.7) +
    geom_text(aes(label = paste0(round(Percentage, 1), "%")), vjust = -0.5, size = 5) +
    scale_fill_manual(values = my_colors) +
    theme_classic() +
    ggtitle(paste0(celltype_prefix, ": % Classified as Injured")) +
    ylab("% Exceeding Normal Limit")

  # Keep DAMN in output filename for downstream compatibility.
  dist_filename <- paste0(celltype_prefix, "_Density_DAMN_Score_Distribution")
  tryCatch({
    save_pub_plot(p_dens, dist_filename, save_dir_dist, w = 6, h = 4.5)
  }, error = function(e) message("Note: save_pub_plot failed or is missing. Please save p_dens manually if needed."))

  # --- 9. Statistical Testing ---
  message("\n--- Statistical Analysis ---")
  k_test <- kruskal.test(Delta ~ Status, data = plot_df)
  message(paste0("Global Difference in Injury Scores (Kruskal-Wallis): p = ", format(k_test$p.value, digits = 3)))

  pair_test <- pairwise.wilcox.test(plot_df$Delta, plot_df$Status, p.adjust.method = "bonferroni")
  message("\nPairwise Comparisons (Score Severity):")
  print(pair_test)

  cont_table <- table(plot_df$Status, plot_df$mouse_DAMN_State)
  print(cont_table)
  chi_test <- chisq.test(cont_table)
  message(paste0("\nDifference in Proportions (Chi-Squared): p = ", format(chi_test$p.value, digits = 3)))


  # DEBUG BLOCK:

  
  # --- 10. Subtype LMM ---
  model_data <- merged_obj@meta.data %>%
    dplyr::select(Status, SampleID, Score_DAMN_Delta)

  lmm_model <- lmer(Score_DAMN_Delta ~ Status + (1 | SampleID), data = model_data)
  emm <- emmeans(
  lmm_model,
  specs = pairwise ~ Status,
  lmerTest.limit = nrow(model_data)
)


  plot_stats <- model_data %>%
    group_by(Status, SampleID) %>%
    summarise(Mean_Score = mean(Score_DAMN_Delta, na.rm = TRUE), Cell_Count = n(), .groups = "drop")

  max_cells <- max(plot_stats$Cell_Count)
  my_breaks <- c(1, 10, 50, 100, 250, 500, 800)
  my_breaks <- my_breaks[my_breaks <= max_cells + 100] 

  p_vals <- as.data.frame(emm$contrasts)
  get_pval <- function(comparison) {
    idx <- which(grepl("Control", p_vals$contrast) & grepl(comparison, p_vals$contrast))
    if (length(idx) > 0) {
      p <- p_vals$p.value[idx]
      if (p < 0.001) return("< 0.001") else return(format(p, digits = 3))
    } else {
      return("NA")
    }
  }

  p_lmm <- ggplot(plot_stats, aes(x = Status, y = Mean_Score, fill = Status)) +
    geom_boxplot(alpha = 0.4, outlier.shape = NA) +
    geom_jitter(aes(size = Cell_Count), width = 0.15, shape = 21, color = "black") +
    scale_fill_manual(values = my_colors) +
    scale_size_continuous(
      range = c(2, 8),
      name = obj_type,
      breaks = my_breaks,
      limits = c(min(plot_stats$Cell_Count), max(plot_stats$Cell_Count))
    ) +
    theme_classic(base_size = 22) +
    theme(axis.title.y = element_text(size = 20)) + 
    labs(
      fill = NULL,
      y = "Mean Weighted DM Score",
      x = "",
      caption = paste0(
        "LMM P-values vs Control:\n",
        "SOD1: p=", get_pval("SOD1"),
        "\nC9: p=", get_pval("C9orf72"),
        "\nsALS: p=", get_pval("sALS")
      )
    )

  out_file_lmm <- file.path(save_dir_lmm, paste0(celltype_prefix, "_LMM_Weighted_Injury_Stats_with_ALS.png"))
  ggsave(out_file_lmm, p_lmm, width = 7, height = 6)

  # --- 11. Pooled LMM ---
  model_data$Pooled_Condition <- ifelse(model_data$Status == "Control", "Control", "All_ALS")
  model_data$Pooled_Condition <- factor(model_data$Pooled_Condition, levels = c("Control", "All_ALS"))

  lmm_pooled <- lmer(Score_DAMN_Delta ~ Pooled_Condition + (1 | SampleID), data = model_data)
  emm_pooled <- emmeans(lmm_pooled, specs = pairwise ~ Pooled_Condition)
  
  p_val_raw <- as.data.frame(emm_pooled$contrasts)$p.value
  p_val_str <- format(p_val_raw, digits = 3, scientific = TRUE)

  plot_stats_pooled <- model_data %>%
    group_by(Pooled_Condition, Status, SampleID) %>%
    summarise(Mean_Score = mean(Score_DAMN_Delta, na.rm = TRUE), Cell_Count = n(), .groups = "drop")

  p_pooled <- ggplot(plot_stats_pooled, aes(x = Pooled_Condition, y = Mean_Score)) +
    geom_boxplot(aes(fill = Pooled_Condition), alpha = 0.2, outlier.shape = NA, width = 0.5) +
    geom_jitter(aes(size = Cell_Count, fill = Status), width = 0.2, shape = 21, color = "black", stroke = 0.5) +
    scale_fill_manual(values = c(
      "Control" = "grey70",
      "All_ALS" = "firebrick",
      "SOD1" = "firebrick",
      "C9orf72" = "orange",
      "sALS" = "steelblue"
    )) +
    scale_size_continuous(
      range = c(2, 8),
      name = obj_type,
      breaks = my_breaks,
      limits = c(min(plot_stats_pooled$Cell_Count), max(plot_stats_pooled$Cell_Count))
    ) +
    theme_classic(base_size = 22) +
    theme(axis.title.y = element_text(size = 20)) + 
    labs(
      fill = NULL,
      subtitle = paste0("LMM p-value: ", p_val_str),
      y = "Mean Weighted DM Score",
      x = ""
    ) 

  out_file_pooled <- file.path(save_dir_lmm, paste0(celltype_prefix, "_LMM_Pooled_ALS_Stats.png"))
  ggsave(out_file_pooled, p_pooled, width = 6, height = 6)
  
  return(merged_obj)
}


In [ ]:
# Define your directories based on your project structure
# --- 0. Directory Setup -------------------------------------------------------
# Set your main publication directory (matches your previous scripts)
if(!exists("pub_dir")) pub_dir <- "Plots" 

# Dynamically route the subfolders
distribution_dir <- file.path(pub_dir, "1_Distributions") 
lmm_dir <- file.path(pub_dir, "3_LMM_Stats")

# Make sure the folders actually exist before the pipeline tries to save to them!
dir.create(distribution_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(lmm_dir, recursive = TRUE, showWarnings = FALSE)


# --- 0. Directory Setup (Routed to Nuclei Subfolders) -------------------------
if(!exists("pub_dir")) pub_dir <- "Plots" 

# Dynamically route the subfolders into a "Nuclei" specific directory
distribution_dir <- file.path(pub_dir, "1_Distributions", "Nuclei") 
lmm_dir <- file.path(pub_dir, "3_LMM_Stats", "Nuclei")

# Make sure the folders actually exist
dir.create(distribution_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(lmm_dir, recursive = TRUE, showWarnings = FALSE)

# Ensure the active identity is set to our custom cell types
Idents(nuclei_mn) <- "MN_Type"


# --- 1. Run Alpha Motor Neurons (Clusters 1, 4) -------------------------------
alphamn_analyzed <- run_damn_pipeline(
  seurat_obj = subset(nuclei_mn, idents = "Alpha MNs"),
  celltype_prefix = "Nuclei_AlphaMN",
  conditions_to_include = c("SOD1", "ctrl", "C9", "sALS"),
  pvalue_threshold = 0.01,
  top_n_genes = 1000,
  save_dir_dist = distribution_dir,
  save_dir_lmm = lmm_dir,
  signature_csv_path = "damn_signature.csv",
  obj_type= "n Nuclei"
)

# --- 2. Run Putative Alpha Motor Neurons (Cluster 11) -------------------------
putative_alpha_analyzed <- run_damn_pipeline(
  seurat_obj = subset(nuclei_mn, idents = "ATF3+ Alpha MNs"),
  celltype_prefix = "Nuclei_PutativeAlphaMN",
  conditions_to_include = c("SOD1", "ctrl", "C9", "sALS"),
  pvalue_threshold = 0.01,
  top_n_genes = 1000,
  save_dir_dist = distribution_dir,
  save_dir_lmm = lmm_dir,
  signature_csv_path = "damn_signature.csv",
  obj_type= "n Nuclei"

)

# --- 3. Run Gamma Motor Neurons (Clusters 0, 8) -------------------------------
gammamn_analyzed <- run_damn_pipeline(
  seurat_obj = subset(nuclei_mn, idents = "Gamma MNs"),
  celltype_prefix = "Nuclei_GammaMN",
  conditions_to_include = c("SOD1", "ctrl", "C9", "sALS"),
  pvalue_threshold = 0.01,
  top_n_genes = 1000,
  save_dir_dist = distribution_dir,
  save_dir_lmm = lmm_dir,
  signature_csv_path = "damn_signature.csv",
  obj_type= "n Nuclei"

)

# --- 4. Run Putative Gamma* Motor Neurons (Cluster 10) ------------------------
putative_gamma_analyzed <- run_damn_pipeline(
  seurat_obj = subset(nuclei_mn, idents = "Gamma* MNs"),
  celltype_prefix = "Nuclei_PutativeGammaStarMN",
  conditions_to_include = c("SOD1", "ctrl", "C9", "sALS"),
  pvalue_threshold = 0.01,
  top_n_genes = 1000,
  save_dir_dist = distribution_dir,
  save_dir_lmm = lmm_dir,
  signature_csv_path = "damn_signature.csv",
  obj_type= "n Nuclei"

)

# --- 5. Run Visceral Motor Neurons (Clusters 2, 3, 5, 6, 7, 9) ----------------
visceral_analyzed <- run_damn_pipeline(
  seurat_obj = subset(nuclei_mn, idents = "Visceral MNs"),
  celltype_prefix = "Nuclei_VisceralMN",
  conditions_to_include = c("SOD1", "ctrl", "C9", "sALS"),
  pvalue_threshold = 0.01,
  top_n_genes = 1000,
  save_dir_dist = distribution_dir,
  save_dir_lmm = lmm_dir,
  signature_csv_path = "damn_signature.csv",
  obj_type= "n Nuclei"

)

In [ ]:
# --- 0. Directory Setup (Routed to Debris Subfolders) -------------------------
if(!exists("pub_dir")) pub_dir <- "Plots" 

# Dynamically route the subfolders into a "Debris" specific directory
distribution_dir <- file.path(pub_dir, "1_Distributions", "Debris") 
lmm_dir <- file.path(pub_dir, "3_LMM_Stats", "Debris")

# Make sure the folders actually exist before the pipeline tries to save to them!
dir.create(distribution_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(lmm_dir, recursive = TRUE, showWarnings = FALSE)

# --- 1. Iterate through target_objects ----------------------------------------
# Create an empty list to store the analyzed objects in your R environment
debris_analyzed_objects <- list()

for (label in names(target_objects)) {
  
  message(paste0("\n========================================================"))
  message(paste0(" Running DM Pipeline: ", label))
  message(paste0("========================================================"))
  
  # Dynamically build the prefix (e.g., "Debris_AlphaMN")
  current_prefix <- paste0("Debris_", label)
  
  # Run the pipeline and store the result in our new list
  debris_analyzed_objects[[label]] <- run_damn_pipeline(
    seurat_obj = target_objects[[label]],
    celltype_prefix = current_prefix,
    conditions_to_include = c("SOD1", "ctrl", "C9", "ALS"), # Note: I used "ALS" here to match your DESeq2 code, change to "sALS" if your metadata differs here
    pvalue_threshold = 0.01,
    top_n_genes = 1000,
    save_dir_dist = distribution_dir,
    save_dir_lmm = lmm_dir,
    signature_csv_path = "damn_signature.csv",
    obj_type = "n Frag"
  )
}

message("\nPipeline complete for all Debris target objects!")